In [1]:
#Cell 1
!pip install --quiet yfinance prophet statsmodels scikit-learn torch torchvision newsapi-python textblob
!python -m textblob.download_corpora

from textblob import TextBlob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.9 MB/s eta 0:00:00
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloa

In [2]:
#Cell 2
import os, getpass

# Check if key already exists (useful for re-running cells)
if not os.environ.get('NEWS_API_KEY'):
    try:
        key = getpass.getpass("🔑 Enter your NewsAPI key: ")
        if key.strip():
            os.environ['NEWS_API_KEY'] = key
            print("✅ NewsAPI key set successfully")
        else:
            print("⚠️ No key provided - sentiment analysis will use neutral fallback")
    except KeyboardInterrupt:
        print("⚠️ Key input cancelled - sentiment analysis will use neutral fallback")
else:
    print("✅ NewsAPI key already configured")

🔑 Enter your NewsAPI key: ··········
✅ NewsAPI key set successfully


In [3]:
# 📦 MCP-Tradi-Win - Cell 3: Enhanced Context Schema + AgentBase + ForecastAgent + Advanced SentimentAgent
# This cell defines the shared protocol context and initializes the agent-based architecture.

# -----------------------------
# ✅ Enhanced Shared Context Schema (protocol-wide)
# -----------------------------
context = {
    "data": {
        "symbol": "BTC",
        "raw_prices": None,
        "timestamp": None
    },
    "forecast": {
        "model_used": None,
        "predicted_movement": None,  # "up", "down", or "stable"
        "confidence": None,
        "metrics_per_model": {}
    },
    "sentiment": {
        "enabled": False,
        "source": None,
        "score": None,
        "verdict": None,  # "bullish", "bearish", or "neutral"
        "confidence": None,
        # ✅ NEW ADVANCED METRICS:
        "strength_category": None,      # "Weak", "Moderate", "Strong"
        "news_volume": None,           # Number of articles found
        "recency_score": None,         # How fresh the news is (0-1)
        "directional_accuracy": None,  # Historical hit rate for sentiment
        "used_in_decision": False,
        "alignment": None,
        "headlines_sample": []         # Sample headlines for transparency
    },
    "decision": {
        "action": None,
        "rationale": None
    },
    "history": [],
    "sentiment_history": [],  # ✅ NEW: Track sentiment predictions vs actual outcomes
    "explainability": {
        "enabled": True,
        "explanation_type": "template",
        "full_trace": []
    }
}

# -----------------------------
# ✅ Simplified MCP Design - Using OrchestratorChain (Cell 6)
# -----------------------------
# Note: Complex orchestrator removed in favor of practical OrchestratorChain in Cell 6

# -----------------------------
# ✅ Agent Interface
# -----------------------------
class AgentBase:
    def run(self, context):
        raise NotImplementedError("Each agent must implement the run(context) method.")

# -----------------------------
# 🔮 Enhanced ForecastAgent: ARIMA, Prophet, LSTM with Multi-Metric Selection
# -----------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

class ForecastAgent(AgentBase):
    def __init__(self):
        # Session-based model caching
        self._cached_models = {}
        self._cache_timestamp = None

    def calculate_directional_accuracy(self, y_true, y_pred):
        """Calculate how often we predict direction correctly"""
        if len(y_true) < 2 or len(y_pred) < 2:
            return 0.0

        actual_direction = np.sign(np.diff(y_true))
        predicted_direction = np.sign(np.diff(y_pred))

        # Handle case where lengths don't match
        min_len = min(len(actual_direction), len(predicted_direction))
        if min_len == 0:
            return 0.0

        matches = (actual_direction[:min_len] == predicted_direction[:min_len]).sum()
        return float(matches / min_len)

    def composite_score(self, metrics, latest_price):
        """Combine Directional Accuracy (40%), MAPE (30%), and MAE (30%) for model selection"""
        try:
            # Normalize MAE by price to make it scale-independent
            normalized_mae = metrics["MAE"] / max(latest_price, 1e-8)

            # Convert to "goodness" scores (higher = better)
            dir_acc_score = metrics["Directional_Accuracy"]  # Already 0-1, higher is better
            mape_score = max(0, 1 - metrics["MAPE"] / 100)  # Convert MAPE to goodness score
            mae_score = max(0, 1 - normalized_mae)  # Convert normalized MAE to goodness score

            # Weighted combination: 40% Dir Acc, 30% MAPE, 30% MAE
            composite = 0.4 * dir_acc_score + 0.3 * mape_score + 0.3 * mae_score
            return composite
        except:
            return 0.0

    def run(self, context):
        symbol = context["data"]["symbol"]

        # Enhanced data fetching: 120 days instead of 90
        df = yf.download(tickers=symbol + "-USD", period="120d", interval="1d", progress=False)

        if df.empty or "Close" not in df:
            context["forecast"]["model_used"] = "none"
            context["forecast"]["predicted_movement"] = "error"
            context["forecast"]["confidence"] = 0.0
            context["explainability"]["full_trace"].append(
                "[ForecastAgent] Data fetch failed. No forecast generated."
            )
            return context

        # ✅ Assign the correct date
        context["data"]["timestamp"] = df.index[-1].strftime("%Y-%m-%d")
        print("✅ Latest Fetched Date:", context["data"]["timestamp"])

        df['ds'] = df.index
        df['y'] = df['Close']

        # Enhanced data window: 45 days instead of 30 for better patterns
        prices = df['Close'].values[-45:]
        latest_price = float(prices[-1])

        results = {}
        metrics_per_model = {}

        # === ARIMA ===
        try:
            arima_model = ARIMA(prices, order=(3,1,0)).fit()
            arima_forecast = float(arima_model.forecast()[0])
            arima_pred = arima_model.predict()

            # Use more data points for validation (last 10 instead of 5)
            validation_size = min(10, len(arima_pred))
            arima_rmse = float(np.sqrt(mean_squared_error(prices[-validation_size:], arima_pred[-validation_size:])))
            arima_mae = float(mean_absolute_error(prices[-validation_size:], arima_pred[-validation_size:]))

            # ✅ MAPE with epsilon to avoid division by zero / near-zero
            eps = 1e-8
            den = np.maximum(np.abs(prices[-validation_size:]), eps)
            arima_mape = float(np.mean(np.abs((prices[-validation_size:] - arima_pred[-validation_size:]) / den)) * 100)

            # ✅ Directional Accuracy
            arima_directional = self.calculate_directional_accuracy(prices[-validation_size:], arima_pred[-validation_size:])

            results["ARIMA"] = arima_forecast
            metrics_per_model["ARIMA"] = {
                "RMSE": arima_rmse,
                "MAE": arima_mae,
                "MAPE": arima_mape,
                "Directional_Accuracy": arima_directional
            }
        except Exception as e:
            context["explainability"]["full_trace"].append(f"[ForecastAgent] ARIMA failed: {e}")

        # === Prophet ===
        try:
            prophet = Prophet()
            prophet.fit(df[['ds', 'y']])
            future = prophet.make_future_dataframe(periods=1)
            forecast = prophet.predict(future)
            prophet_forecast = float(forecast['yhat'].iloc[-1])

            validation_size = min(10, len(forecast) - 1)
            y_true = np.array(df['y'][-validation_size:], dtype=float)
            y_pred = np.array(forecast['yhat'][-(validation_size+1):-1], dtype=float)

            prophet_rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
            prophet_mae = float(mean_absolute_error(y_true, y_pred))

            # ✅ MAPE with epsilon
            eps = 1e-8
            den = np.maximum(np.abs(y_true), eps)
            prophet_mape = float(np.mean(np.abs((y_true - y_pred) / den)) * 100)

            # ✅ Directional Accuracy
            prophet_directional = self.calculate_directional_accuracy(y_true, y_pred)

            results["Prophet"] = prophet_forecast
            metrics_per_model["Prophet"] = {
                "RMSE": prophet_rmse,
                "MAE": prophet_mae,
                "MAPE": prophet_mape,
                "Directional_Accuracy": prophet_directional
            }
        except Exception as e:
            context["explainability"]["full_trace"].append(f"[ForecastAgent] Prophet failed: {e}")

        # === Enhanced LSTM (Returns-based, Option B) ===
        try:
            # Convert prices to percentage returns
            returns = np.diff(prices) / prices[:-1] * 100  # Percentage returns

            # Scale returns for LSTM
            scaler = MinMaxScaler(feature_range=(-1, 1))
            scaled_returns = scaler.fit_transform(returns.reshape(-1, 1)).flatten()

            # Create sequences with longer lookback (12 days instead of 5)
            lookback = 12
            X, y = [], []
            for i in range(len(scaled_returns) - lookback):
                X.append(scaled_returns[i:i+lookback])
                y.append(scaled_returns[i+lookback])
            X, y = np.array(X), np.array(y)

            if len(X) > 0:
                # Enhanced LSTM architecture with dropout
                model = Sequential()
                model.add(LSTM(50, return_sequences=True, input_shape=(lookback, 1)))
                model.add(Dropout(0.2))
                model.add(LSTM(25, return_sequences=False))
                model.add(Dropout(0.2))
                model.add(Dense(1))
                model.compile(optimizer='adam', loss='mse')

                # Train with validation split
                X_reshaped = X.reshape(X.shape[0], X.shape[1], 1)
                model.fit(X_reshaped, y, epochs=20, verbose=0, validation_split=0.2)

                # Predict next return
                last_sequence = scaled_returns[-lookback:].reshape(1, lookback, 1)
                predicted_return_scaled = model.predict(last_sequence, verbose=0)[0][0]
                predicted_return = scaler.inverse_transform([[predicted_return_scaled]])[0][0]

                # Convert predicted return back to price
                lstm_forecast = float(latest_price * (1 + predicted_return / 100))

                # Calculate metrics on validation set
                val_size = min(10, len(X))
                y_pred_scaled = model.predict(X_reshaped[-val_size:], verbose=0).flatten()
                y_pred_returns = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
                y_true_returns = scaler.inverse_transform(y[-val_size:].reshape(-1, 1)).flatten()

                # Convert back to prices for metrics calculation
                base_prices = prices[-(val_size+1):-1]
                y_pred_prices = base_prices * (1 + y_pred_returns / 100)
                y_true_prices = base_prices * (1 + y_true_returns / 100)

                lstm_rmse = float(np.sqrt(mean_squared_error(y_true_prices, y_pred_prices)))
                lstm_mae = float(mean_absolute_error(y_true_prices, y_pred_prices))

                # ✅ MAPE with epsilon
                eps = 1e-8
                den = np.maximum(np.abs(y_true_prices), eps)
                lstm_mape = float(np.mean(np.abs((y_true_prices - y_pred_prices) / den)) * 100)

                # ✅ Directional Accuracy
                lstm_directional = self.calculate_directional_accuracy(y_true_prices, y_pred_prices)

                results["LSTM"] = lstm_forecast
                metrics_per_model["LSTM"] = {
                    "RMSE": lstm_rmse,
                    "MAE": lstm_mae,
                    "MAPE": lstm_mape,
                    "Directional_Accuracy": lstm_directional
                }
            else:
                raise ValueError("Insufficient data for LSTM training")

        except Exception as e:
            context["explainability"]["full_trace"].append(f"[ForecastAgent] LSTM failed: {e}")

        # === Enhanced Model Selection using Composite Score ===
        if not results or not metrics_per_model:
            context["forecast"]["model_used"] = "none"
            context["forecast"]["predicted_movement"] = "error"
            context["forecast"]["confidence"] = 0.0
            context["explainability"]["full_trace"].append("[ForecastAgent] All models failed.")
            return context

        # Select best model using composite score
        model_scores = {}
        for model_name, metrics in metrics_per_model.items():
            score = self.composite_score(metrics, latest_price)
            model_scores[model_name] = score

        best_model = max(model_scores.items(), key=lambda x: x[1])[0]
        forecast_value = results[best_model]
        best_metrics = metrics_per_model[best_model]

        # Calculate movement and confidence
        movement = "up" if forecast_value > latest_price else "down" if forecast_value < latest_price else "stable"

        # Enhanced confidence calculation using directional accuracy
        directional_conf = best_metrics["Directional_Accuracy"]
        rmse_conf = max(0.0, min(1.0, 1 - (best_metrics["RMSE"] / max(latest_price, 1e-8))))
        confidence = round((0.6 * directional_conf + 0.4 * rmse_conf), 3)

        # === Update context ===
        context["forecast"]["model_used"] = best_model
        context["forecast"]["predicted_movement"] = movement
        context["forecast"]["confidence"] = confidence
        context["forecast"]["metrics_per_model"] = metrics_per_model

        context["explainability"]["full_trace"].append(
            f"[ForecastAgent] {best_model} selected (composite score: {model_scores[best_model]:.3f}) "
            f"predicts {movement.upper()} (conf={confidence:.3f}). "
            f"Metrics: RMSE=${best_metrics['RMSE']:.2f}, MAE=${best_metrics['MAE']:.2f}, "
            f"MAPE={best_metrics['MAPE']:.1f}%, Dir_Acc={best_metrics['Directional_Accuracy']:.1%}"
        )
        return context

# -----------------------------
# 🧠 Enhanced SentimentAgent: NewsAPI + TextBlob + Advanced Metrics
# -----------------------------
from textblob import TextBlob
from newsapi import NewsApiClient
from datetime import datetime as _dt, timedelta
import re

class SentimentAgent(AgentBase):
    def __init__(self):
        api_key = os.environ.get('NEWS_API_KEY')
        if not api_key:
            raise ValueError("🛑 NEWS_API_KEY not found in environment variables.")
        self.newsapi = NewsApiClient(api_key=api_key)

    def calculate_sentiment_directional_accuracy(self, context):
        """Calculate historical sentiment directional accuracy"""
        sentiment_history = context.get("sentiment_history", [])

        if len(sentiment_history) < 2:
            return 0.65  # Default reasonable accuracy for bootstrap

        # Count correct directional predictions
        correct_predictions = 0
        total_predictions = 0

        for entry in sentiment_history[-10:]:  # Last 10 predictions
            if all(k in entry for k in ["sentiment_direction", "actual_direction"]):
                total_predictions += 1
                if entry["sentiment_direction"] == entry["actual_direction"]:
                    correct_predictions += 1

        if total_predictions == 0:
            return 0.65  # Default

        return round(correct_predictions / total_predictions, 3)

    def categorize_sentiment_strength(self, polarity_score):
        """Categorize sentiment strength based on absolute polarity"""
        abs_score = abs(polarity_score)
        if abs_score >= 0.6:
            return "Strong"
        elif abs_score >= 0.2:
            return "Moderate"
        else:
            return "Weak"

    def calculate_recency_score(self, articles):
        """Calculate how fresh the news is (0-1 scale)"""
        if not articles:
            return 0.0

        now = _dt.now()
        recency_scores = []

        for article in articles:
            published_at = article.get('publishedAt')
            if not published_at:
                continue

            try:
                # Parse ISO format: 2024-01-15T10:30:00Z
                pub_time = _dt.fromisoformat(published_at.replace('Z', '+00:00'))
                pub_time = pub_time.replace(tzinfo=None)  # Remove timezone for comparison

                # Calculate hours ago
                hours_ago = (now - pub_time).total_seconds() / 3600

                # Score: 1.0 for very recent (0-6hrs), decaying to 0.1 for old (48+ hrs)
                if hours_ago <= 6:
                    score = 1.0
                elif hours_ago <= 24:
                    score = 0.8
                elif hours_ago <= 48:
                    score = 0.5
                else:
                    score = 0.1

                recency_scores.append(score)
            except:
                recency_scores.append(0.3)  # Default for parsing errors

        return round(sum(recency_scores) / len(recency_scores) if recency_scores else 0.3, 3)

    def run(self, context):
        if not context["sentiment"]["enabled"]:
            return context

        # ✅ Dynamic crypto symbol mapping for news search
        symbol = context["data"]["symbol"]
        crypto_search_terms = {
            "BTC": "bitcoin",
            "ETH": "ethereum",
            "SOL": "solana",
            "ADA": "cardano",
            "XRP": "ripple",
            "BNB": "binance coin",
            "MATIC": "polygon",
            "DOT": "polkadot",
            "LINK": "chainlink",
            "AVAX": "avalanche",
            "UNI": "uniswap",
            "LTC": "litecoin",
            "ATOM": "cosmos",
            "FTM": "fantom",
            "ALGO": "algorand"
        }
        search_term = crypto_search_terms.get(symbol, symbol.lower())

        try:
            articles = self.newsapi.get_everything(
                q=search_term,
                language='en',
                sort_by='publishedAt',
                page_size=10  # Increased for better volume analysis
            )
        except Exception as e:
            context["sentiment"].update({
                "source": "NewsAPI",
                "score": 0.0,
                "verdict": "neutral",
                "confidence": 0.0,
                "strength_category": "Weak",
                "news_volume": 0,
                "recency_score": 0.0,
                "directional_accuracy": 0.65,
                "headlines_sample": []
            })
            context["explainability"]["full_trace"].append(
                f"[SentimentAgent] Error fetching '{search_term}' headlines: {e}. Defaulting to NEUTRAL."
            )
            return context

        articles_list = articles.get('articles', [])
        headlines = [a['title'] for a in articles_list if a.get('title')]

        if not headlines:
            context["sentiment"].update({
                "source": "NewsAPI",
                "score": 0.0,
                "verdict": "neutral",
                "confidence": 0.0,
                "strength_category": "Weak",
                "news_volume": 0,
                "recency_score": 0.0,
                "directional_accuracy": 0.65,
                "headlines_sample": []
            })
            context["explainability"]["full_trace"].append(
                f"[SentimentAgent] No '{search_term}' headlines found. Defaulting to NEUTRAL."
            )
            return context

        # ✅ ENHANCED SENTIMENT ANALYSIS

        # Basic sentiment analysis
        polarities = [TextBlob(title).sentiment.polarity for title in headlines]
        avg_polarity = float(sum(polarities) / len(polarities))
        std_dev = float(np.std(polarities))

        # ✅ NEW METRICS CALCULATION

        # 1. Strength categorization
        strength_category = self.categorize_sentiment_strength(avg_polarity)

        # 2. News volume
        news_volume = len(headlines)

        # 3. Recency score
        recency_score = self.calculate_recency_score(articles_list)

        # 4. Historical directional accuracy
        directional_accuracy = self.calculate_sentiment_directional_accuracy(context)

        # 5. Enhanced confidence calculation
        base_confidence = 1.0 - std_dev  # Agreement between headlines
        volume_boost = min(0.2, news_volume / 50)  # More articles = higher confidence
        recency_boost = recency_score * 0.15  # Fresh news = higher confidence

        enhanced_confidence = round(
            0.6 * base_confidence + 0.3 * volume_boost + 0.1 * recency_boost, 3
        )
        enhanced_confidence = max(0.0, min(1.0, enhanced_confidence))

        # Verdict determination
        if avg_polarity > 0.2:
            verdict = "bullish"
        elif avg_polarity < -0.2:
            verdict = "bearish"
        else:
            verdict = "neutral"

        # Sample headlines for transparency
        headlines_sample = headlines[:3]  # Show first 3 headlines

        # ✅ UPDATE CONTEXT WITH ALL NEW METRICS
        context["sentiment"].update({
            "source": "NewsAPI",
            "score": round(avg_polarity, 3),
            "verdict": verdict,
            "confidence": enhanced_confidence,
            "strength_category": strength_category,
            "news_volume": news_volume,
            "recency_score": recency_score,
            "directional_accuracy": directional_accuracy,
            "used_in_decision": False,  # updated by DecisionAgent
            "alignment": None,
            "headlines_sample": headlines_sample
        })

        if context["explainability"]["enabled"]:
            context["explainability"]["full_trace"].append(
                f"[SentimentAgent] Analyzed {news_volume} '{search_term}' headlines: "
                f"Avg polarity = {avg_polarity:.3f} ({strength_category} strength), "
                f"Recency = {recency_score:.1%}, Dir_Acc = {directional_accuracy:.1%} "
                f"→ {verdict.upper()} (Confidence={enhanced_confidence:.1%})"
            )
        return context

# -----------------------------
# ✅ DecisionAgent: symmetric buy/sell + sentiment-adjusted thresholds
# -----------------------------
class DecisionAgent(AgentBase):
    def run(self, context):
        forecast = context["forecast"]
        sentiment = context["sentiment"]

        action = "hold"
        rationale = ""
        used_sentiment = bool(sentiment.get("enabled", False))
        aligned = None

        move = forecast.get("predicted_movement", "stable")   # "up" / "down" / "stable"
        conf = float(forecast.get("confidence", 0.0))
        sv = sentiment.get("verdict", "neutral") if used_sentiment else "neutral"  # "bullish"/"bearish"/"neutral"

        # Base threshold
        THRESH = 0.60
        # Sentiment-adjusted thresholds
        if move == "up":
            aligned = (sv == "bullish")
            thr = THRESH - 0.10 if aligned else (THRESH + 0.10 if sv == "bearish" else THRESH)
        elif move == "down":
            aligned = (sv == "bearish")
            thr = THRESH - 0.10 if aligned else (THRESH + 0.10 if sv == "bullish" else THRESH)
        else:  # "stable"
            aligned = None
            thr = 0.75  # be conservative when model says "stable"

        # Decision
        if move == "up" and conf >= thr:
            action = "buy"
            rationale = f"Forecast {move.upper()} with confidence {conf:.3f} (threshold {thr:.2f}); sentiment={sv}."
        elif move == "down" and conf >= thr:
            action = "sell"
            rationale = f"Forecast {move.upper()} with confidence {conf:.3f} (threshold {thr:.2f}); sentiment={sv}."
        else:
            reason = "confidence too low" if conf < thr else "no directional edge"
            action = "hold"
            rationale = f"Holding: {reason} (conf {conf:.3f} vs thresh {thr:.2f}); sentiment={sv}."

        # ✅ Save result
        context["decision"]["action"] = action
        context["decision"]["rationale"] = rationale

        # ✅ Mark sentiment usage + expose alignment for UI/prints
        context["sentiment"]["used_in_decision"] = used_sentiment
        context["sentiment"]["alignment"] = aligned

        # ✅ Store correlation for analysis
        context["history"].append({
            "timestamp": context["data"]["timestamp"],
            "forecast_movement": move,
            "forecast_confidence": conf,
            "sentiment_verdict": sv,
            "sentiment_score": sentiment.get("score"),
            "sentiment_confidence": sentiment.get("confidence"),
            "sentiment_strength": sentiment.get("strength_category"),
            "news_volume": sentiment.get("news_volume"),
            "recency_score": sentiment.get("recency_score"),
            "alignment": aligned,
            "used_sentiment": used_sentiment
        })

        # ✅ Explainability trace
        if context["explainability"]["enabled"]:
            context["explainability"]["full_trace"].append(
                f"[DecisionAgent] Action: {action} → {rationale}"
            )
        return context

# -----------------------------
# 🔧 Legacy helpers (kept for compatibility)
# -----------------------------
from datetime import datetime

def fetch_price_data(context, symbol="BTC"):
    df = yf.download(tickers=symbol + "-USD", period="120d", interval="1d", progress=False)
    context["data"]["symbol"] = symbol
    context["data"]["df"] = df
    context["data"]["timestamp"] = df.index[-1].strftime("%Y-%m-%d")
    context["explainability"]["full_trace"].append(
        f"[Data Fetch] Loaded price data for {symbol} at {context['data']['timestamp']}"
    )
    return context

def finalize_and_log(context):
    context["history"].append({
        "timestamp": context["data"].get("timestamp", str(datetime.today().date())),
        "decision": context.get("decision", {}),
        "forecast": context.get("forecast", {}),
        "sentiment": context["sentiment"] if context.get("sentiment", {}).get("enabled") else "N/A",
        "explanation_trace": context["explainability"].get("full_trace", [])
    })
    return context

In [4]:

# 📦 MCP-Tradi-Win - Cell 4: Simplified Enhanced Pipeline Runner
# Fallback runner for direct agent execution without orchestrator

# -----------------------------
# 🎯 Enhanced Pipeline Runner (Simplified - No Complex Orchestrator)
# -----------------------------
def MCP_tradi_win_runner(context, use_sentiment=False, model_preference=None):
    """
    Simplified MCP-Tradi-Win pipeline runner for direct execution:

    Args:
        context: Shared context dictionary
        use_sentiment: Enable sentiment analysis (default: False)
        model_preference: Force specific model selection ('ARIMA', 'Prophet', 'LSTM', or None for auto)

    Returns:
        Updated context with all analysis results and full trace

    Note: For advanced orchestration, use OrchestratorChain in Cell 6
    """

    # Initialize configuration
    context["sentiment"]["enabled"] = use_sentiment
    start_time = pd.Timestamp.now()

    context["explainability"]["full_trace"].append(
        f"[SimplePipeline] Starting analysis at {start_time.strftime('%H:%M:%S')} "
        f"(Sentiment: {'Enabled' if use_sentiment else 'Disabled'})"
    )

    try:
        # Step 1: Data fetching with error handling
        try:
            context = fetch_price_data(context)
            context["explainability"]["full_trace"].append(
                "[SimplePipeline] ✅ Data fetch completed successfully"
            )
        except Exception as e:
            context["explainability"]["full_trace"].append(
                f"[SimplePipeline] ❌ Data fetch failed: {e}"
            )
            return context

        # Step 2: Forecasting with error handling
        try:
            forecast_agent = ForecastAgent()
            context = forecast_agent.run(context)

            # Apply model preference if specified
            if model_preference and model_preference in context.get("forecast", {}).get("metrics_per_model", {}):
                original_model = context["forecast"]["model_used"]

                # Update forecast results to use preferred model
                metrics = context["forecast"]["metrics_per_model"][model_preference]
                context["forecast"]["model_used"] = model_preference

                # Recalculate confidence for preferred model
                latest_price = 50000  # Approximate for confidence calculation
                directional_conf = metrics["Directional_Accuracy"]
                rmse_conf = max(0.0, min(1.0, 1 - (metrics["RMSE"] / latest_price)))
                new_confidence = round((0.6 * directional_conf + 0.4 * rmse_conf), 3)
                context["forecast"]["confidence"] = new_confidence

                context["explainability"]["full_trace"].append(
                    f"[SimplePipeline] Model preference override: {original_model} → {model_preference} "
                    f"(Dir_Acc: {metrics['Directional_Accuracy']:.1%}, New confidence: {new_confidence:.1%})"
                )

            context["explainability"]["full_trace"].append(
                "[SimplePipeline] ✅ Forecasting completed successfully"
            )
        except Exception as e:
            context["explainability"]["full_trace"].append(
                f"[SimplePipeline] ❌ Forecasting failed: {e}"
            )
            # Set error state but continue
            context["forecast"] = {"model_used": "error", "predicted_movement": "error", "confidence": 0.0}

        # Step 3: Sentiment analysis with error handling
        try:
            if use_sentiment:
                sentiment_agent = SentimentAgent()
                context = sentiment_agent.run(context)
                context["explainability"]["full_trace"].append(
                    "[SimplePipeline] ✅ Sentiment analysis completed successfully"
                )
            else:
                context["explainability"]["full_trace"].append(
                    "[SimplePipeline] ⏭️ Sentiment analysis skipped (disabled)"
                )
        except Exception as e:
            context["explainability"]["full_trace"].append(
                f"[SimplePipeline] ❌ Sentiment analysis failed: {e} - continuing with neutral sentiment"
            )
            # Ensure neutral sentiment for decision making
            context["sentiment"].update({
                "source": "Error_Fallback",
                "score": 0.0,
                "verdict": "neutral",
                "confidence": 0.0,
                "strength_category": "Weak",
                "news_volume": 0,
                "recency_score": 0.0,
                "directional_accuracy": 0.65
            })

        # Step 4: Decision making with error handling
        try:
            decision_agent = DecisionAgent()
            context = decision_agent.run(context)
            context["explainability"]["full_trace"].append(
                "[SimplePipeline] ✅ Decision making completed successfully"
            )
        except Exception as e:
            context["explainability"]["full_trace"].append(
                f"[SimplePipeline] ❌ Decision making failed: {e}"
            )
            # Set safe default decision
            context["decision"] = {"action": "hold", "rationale": f"Error in decision process: {e}"}

    except Exception as e:
        # Global error handler
        context["explainability"]["full_trace"].append(
            f"[SimplePipeline] ❌ Critical pipeline failure: {e}"
        )
        # Ensure minimal viable context
        if "decision" not in context:
            context["decision"] = {"action": "hold", "rationale": "Pipeline error - defaulting to hold"}

    # Final logging and timing
    try:
        context = finalize_and_log(context)
        end_time = pd.Timestamp.now()
        execution_time = (end_time - start_time).total_seconds()

        context["explainability"]["full_trace"].append(
            f"[SimplePipeline] ✅ Analysis completed in {execution_time:.2f}s at {end_time.strftime('%H:%M:%S')} "
            f"→ Action: {context.get('decision', {}).get('action', 'unknown').upper()}"
        )
    except Exception as e:
        context["explainability"]["full_trace"].append(
            f"[SimplePipeline] ⚠️ Logging failed: {e} - but analysis completed"
        )

    return context

# -----------------------------
# 🎲 Quick Test Runner for Development (Updated)
# -----------------------------
def quick_test_runner(symbol="BTC", use_sentiment=False, model_preference=None):
    """
    Quick test function for development and debugging

    Args:
        symbol: Crypto symbol to analyze
        use_sentiment: Enable sentiment analysis
        model_preference: Force specific model ('ARIMA', 'Prophet', 'LSTM')
    """
    test_context = context.copy()
    test_context["data"]["symbol"] = symbol

    print(f"🧪 Quick Test: {symbol} | Sentiment: {use_sentiment} | Model: {model_preference or 'Auto'}")
    print("-" * 70)

    result = MCP_tradi_win_runner(
        test_context,
        use_sentiment=use_sentiment,
        model_preference=model_preference
    )

    # Print key results
    decision = result.get("decision", {})
    forecast = result.get("forecast", {})
    sentiment = result.get("sentiment", {})

    print(f"📊 Model: {forecast.get('model_used', 'N/A')}")
    print(f"📈 Prediction: {forecast.get('predicted_movement', 'N/A')} (confidence: {forecast.get('confidence', 0):.1%})")

    if sentiment.get("enabled"):
        print(f"🧠 Sentiment: {sentiment.get('verdict', 'N/A')} ({sentiment.get('strength_category', 'N/A')})")
        print(f"📰 News: {sentiment.get('news_volume', 0)} articles, {sentiment.get('recency_score', 0):.1%} fresh")

    print(f"💡 Decision: {decision.get('action', 'N/A')}")
    print(f"🧠 Rationale: {decision.get('rationale', 'N/A')}")

    return result

In [5]:

# Cell 5 - Enhanced Demo with Advanced Sentiment Metrics Display

# Install required packages
!pip install tabulate --quiet

from tabulate import tabulate
from datetime import datetime

# ✅ Updated context with today's date
context = {
    "data": {
        "symbol": "BTC",
        "timestamp": datetime.today().strftime("%Y-%m-%d"),  # ✅ today's date
    },
    "forecast": {},
    "sentiment": {
        "enabled": True
    },
    "decision": {},
    "explainability": {
        "enabled": True,              # ✅ ADD THIS LINE
        "full_trace": []
    },
    "history": [],
    "sentiment_history": []  # ✅ NEW: For sentiment tracking
}

# 1. Run the full pipeline
final_context = MCP_tradi_win_runner(context, use_sentiment=True)

# 2. Display final decision
print("══════════════════════════════════════════════════")
print("📈 FINAL TRADING DECISION")
print("══════════════════════════════════════════════════")
print(f"🔹 Action: {final_context['decision']['action'].upper()}")
print(f"🔹 Rationale: {final_context['decision']['rationale']}\n")

# 3. Enhanced forecast info with focus on our key metrics
print("🔍 FORECAST DETAILS")
print(f"🔹 Model Used: {final_context['forecast']['model_used']} (selected by composite score)")
print(f"🔹 Predicted Movement: {final_context['forecast']['predicted_movement'].upper()}")
print(f"🔹 Confidence Score: {final_context['forecast']['confidence']:.1%}")

if "metrics_per_model" in final_context["forecast"]:
    best_model = final_context["forecast"]["model_used"]
    best_metrics = final_context["forecast"]["metrics_per_model"].get(best_model, {})

    print(f"🎯 Key Performance Metrics:")
    print(f"   • Directional Accuracy: {best_metrics.get('Directional_Accuracy', 0):.1%} (most important for trading)")
    print(f"   • MAPE: {best_metrics.get('MAPE', 0):.1f}% (scale-independent error)")
    print(f"   • MAE: ${best_metrics.get('MAE', 0):.2f} (absolute error magnitude)")
    print(f"   • RMSE: ${best_metrics.get('RMSE', 0):.2f} (reference metric)")
    print()

# 4. ✅ ENHANCED SENTIMENT ANALYSIS DISPLAY
if final_context["sentiment"]["enabled"]:
    print("🧠 ADVANCED SENTIMENT ANALYSIS")
    print(f"🔹 Verdict: {final_context['sentiment']['verdict'].upper()} ({final_context['sentiment']['strength_category']} Strength)")
    print(f"🔹 Score: {final_context['sentiment']['score']:.3f} (range: -1 to +1)")

    # ✅ NEW METRICS DISPLAY
    print(f"📊 Sentiment Performance Metrics:")
    print(f"   • News Volume: {final_context['sentiment']['news_volume']} articles analyzed")
    print(f"   • News Freshness: {final_context['sentiment']['recency_score']:.1%} (recent news weight)")
    print(f"   • Historical Accuracy: {final_context['sentiment']['directional_accuracy']:.1%} (directional hit rate)")
    print(f"   • Enhanced Confidence: {final_context['sentiment'].get('confidence', 0):.1%} (volume + agreement + recency)")

    # Alignment display
    alignment_status = final_context['sentiment'].get('alignment')
    if alignment_status is True:
        print(f"🔹 Forecast-Sentiment Alignment: ✅ ALIGNED (threshold boost)")
    elif alignment_status is False:
        print(f"🔹 Forecast-Sentiment Alignment: ❌ CONFLICTING (threshold penalty)")
    else:
        print(f"🔹 Forecast-Sentiment Alignment: ➖ NEUTRAL (no adjustment)")

    print(f"🔹 Used in Decision: {'✅ YES' if final_context['sentiment'].get('used_in_decision') else '❌ NO'}")

    # ✅ SAMPLE HEADLINES FOR TRANSPARENCY
    headlines = final_context['sentiment'].get('headlines_sample', [])
    if headlines:
        print(f"📰 Sample Headlines Analyzed:")
        for i, headline in enumerate(headlines, 1):
            print(f"   {i}. {headline[:80]}{'...' if len(headline) > 80 else ''}")
    print()

# 5. Explainability trace
print("🛡️ EXPLAINABILITY TRACE")
for i, step in enumerate(final_context["explainability"]["full_trace"], 1):
    print(f"{i:02d}. {step}")
print()

# 6. 📊 Enhanced Forecast Metrics Table with Directional Accuracy Focus
if "metrics_per_model" in final_context["forecast"]:
    print("📊 MODEL PERFORMANCE COMPARISON")
    print("🎯 Ranked by Composite Score: 40% Directional Accuracy + 30% MAPE + 30% MAE")

    model_table = []
    selected_model = final_context["forecast"]["model_used"]

    # Calculate composite scores for ranking
    model_scores = []
    for model_name, metrics in final_context["forecast"]["metrics_per_model"].items():
        # Recreate composite score calculation (should match ForecastAgent logic)
        latest_price = 50000  # Approximate, for display purposes
        dir_acc_score = metrics.get("Directional_Accuracy", 0)
        mape_score = max(0, 1 - metrics.get("MAPE", 0) / 100)
        normalized_mae = metrics.get("MAE", 0) / latest_price
        mae_score = max(0, 1 - normalized_mae)
        composite = 0.4 * dir_acc_score + 0.3 * mape_score + 0.3 * mae_score

        model_scores.append((model_name, composite, metrics))

    # Sort by composite score (descending)
    model_scores.sort(key=lambda x: x[1], reverse=True)

    for rank, (model_name, composite_score, metrics) in enumerate(model_scores, 1):
        selected_indicator = "👑" if model_name == selected_model else f"{rank}."
        model_table.append([
            f"{selected_indicator} {model_name}",
            f"{metrics.get('Directional_Accuracy', 0):.1%}",
            f"{metrics.get('MAPE', 0):.1f}%",
            f"${metrics.get('MAE', 0):.2f}",
            f"${metrics.get('RMSE', 0):.2f}",
            f"{composite_score:.3f}"
        ])

    print(tabulate(
        model_table,
        headers=["Model", "Dir Acc ↑", "MAPE ↓", "MAE ↓", "RMSE ↓", "Score ↑"],
        tablefmt="fancy_grid"
    ))
    print("Legend: ↑ = Higher is better, ↓ = Lower is better, 👑 = Selected model")
    print()

# ✅ 7. NEW SENTIMENT PERFORMANCE TABLE
if final_context["sentiment"]["enabled"]:
    print("🧠 SENTIMENT ANALYSIS BREAKDOWN")

    sentiment_performance = [
        ["Verdict", final_context['sentiment']['verdict'].upper()],
        ["Strength Category", final_context['sentiment']['strength_category']],
        ["Polarity Score", f"{final_context['sentiment']['score']:.3f}"],
        ["News Volume", f"{final_context['sentiment']['news_volume']} articles"],
        ["News Freshness", f"{final_context['sentiment']['recency_score']:.1%}"],
        ["Historical Accuracy", f"{final_context['sentiment']['directional_accuracy']:.1%}"],
        ["Enhanced Confidence", f"{final_context['sentiment']['confidence']:.1%}"]
    ]

    print(tabulate(
        sentiment_performance,
        headers=["Metric", "Value"],
        tablefmt="fancy_grid"
    ))
    print()

# 8. 📈 Enhanced Correlation History with New Metrics
if "history" in final_context and final_context["history"]:
    print("📈 COMPREHENSIVE DECISION ANALYSIS HISTORY")
    corr_table = []

    for h in final_context["history"]:
        # Skip any entries missing required fields
        required_keys = ["timestamp", "forecast_movement", "forecast_confidence", "sentiment_verdict"]
        if not all(k in h for k in required_keys):
            continue

        # Format alignment with symbols
        alignment_symbol = "✅" if h.get("alignment") is True else "❌" if h.get("alignment") is False else "➖"

        corr_table.append([
            h["timestamp"],
            h["forecast_movement"].upper(),
            f"{h['forecast_confidence']:.1%}",
            h["sentiment_verdict"].upper(),
            h.get("sentiment_strength", "N/A"),
            h.get("news_volume", "N/A"),
            f"{h.get('recency_score', 0):.1%}",
            alignment_symbol,
            "✅" if h.get("used_sentiment") else "❌"
        ])

    if corr_table:
        print(tabulate(
            corr_table,
            headers=["Date", "Forecast", "F_Conf", "Sentiment", "Strength", "Volume", "Fresh", "Align", "Used"],
            tablefmt="fancy_grid"
        ))
        print("Legend: Align: ✅=Aligned, ❌=Conflicting, ➖=Neutral | Fresh=News Freshness")
    else:
        print("No valid correlation entries available.")

print("\n" + "="*80)
print("🎓 MCP-TRADI-WIN: Explainable Multi-Agent Crypto Trading Assistant")
print("   Enhanced Focus: Directional Accuracy + MAPE + MAE + Advanced Sentiment Metrics")
print("   📊 Forecast: Dir_Acc (40%) + MAPE (30%) + MAE (30%)")
print("   🧠 Sentiment: Volume + Freshness + Historical Accuracy + Agreement")
print("="*80)

✅ Latest Fetched Date: 2025-08-15


INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: /tmp/tmpiog_xh3b/etv9hbk5.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpiog_xh3b/wuouctsf.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.11/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=81049', 'data', 'file=/tmp/tmpiog_xh3b/etv9hbk5.json', 'init=/tmp/tmpiog_xh3b/wuouctsf.json', 'output', 'file=/tmp/tmpiog_xh3b/prophet_modelphw2dgin/prophet_model-20250815035204.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
03:52:04 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
03:52:04 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


══════════════════════════════════════════════════
📈 FINAL TRADING DECISION
══════════════════════════════════════════════════
🔹 Action: SELL
🔹 Rationale: Forecast DOWN with confidence 0.859 (threshold 0.60); sentiment=neutral.

🔍 FORECAST DETAILS
🔹 Model Used: Prophet (selected by composite score)
🔹 Predicted Movement: DOWN
🔹 Confidence Score: 85.9%
🎯 Key Performance Metrics:
   • Directional Accuracy: 77.8% (most important for trading)
   • MAPE: 1.7% (scale-independent error)
   • MAE: $1975.16 (absolute error magnitude)
   • RMSE: $2408.64 (reference metric)

🧠 ADVANCED SENTIMENT ANALYSIS
🔹 Verdict: NEUTRAL (Weak Strength)
🔹 Score: 0.026 (range: -1 to +1)
📊 Sentiment Performance Metrics:
   • News Volume: 10 articles analyzed
   • News Freshness: 50.0% (recent news weight)
   • Historical Accuracy: 65.0% (directional hit rate)
   • Enhanced Confidence: 58.9% (volume + agreement + recency)
🔹 Forecast-Sentiment Alignment: ❌ CONFLICTING (threshold penalty)
🔹 Used in Decision: ✅ YES
📰 

In [6]:

# Cell 6 – Enhanced OrchestratorChain + Advanced Sentiment Persistence
# This cell introduces the practical Agentic AI orchestration layer, replacing complex orchestrators
# with a modular OrchestratorChain that runs each agent (DataFetcher → Forecast → Sentiment → Decision) in sequence.
# Enhanced with advanced sentiment metrics persistence and comprehensive artifact storage.

# === Enhanced OrchestratorChain + DataFetcherAgent + Advanced Persistence ===
import json, os, csv
from datetime import datetime

ART_DIR = "artifacts"
os.makedirs(ART_DIR, exist_ok=True)

# Wrap your existing fetch function as an agent
class DataFetcherAgent(AgentBase):
    def run(self, context):
        return fetch_price_data(context, symbol=context["data"].get("symbol","BTC"))

def persist_artifacts(context):
    """
    Enhanced persistence function with advanced sentiment metrics support
    Saves JSON artifacts and CSV correlation log with all new metrics
    """

    # ✅ Enhanced JSON artifacts with all sentiment metrics
    if context.get("forecast"):
        forecast_data = context["forecast"].copy()
        # Add timestamp for tracking
        forecast_data["timestamp"] = context["data"].get("timestamp")
        forecast_data["symbol"] = context["data"].get("symbol")

        with open(f"{ART_DIR}/forecast_results.json", "w") as f:
            json.dump(forecast_data, f, indent=2)

    if context.get("sentiment"):
        sentiment_data = context["sentiment"].copy()
        # Add timestamp for tracking
        sentiment_data["timestamp"] = context["data"].get("timestamp")
        sentiment_data["symbol"] = context["data"].get("symbol")

        with open(f"{ART_DIR}/sentiment_results.json", "w") as f:
            json.dump(sentiment_data, f, indent=2, default=str)  # default=str for any complex objects

    if context.get("decision"):
        decision_data = context["decision"].copy()
        # Add context for decision tracking
        decision_data["timestamp"] = context["data"].get("timestamp")
        decision_data["symbol"] = context["data"].get("symbol")
        decision_data["forecast_confidence"] = context.get("forecast", {}).get("confidence")
        decision_data["sentiment_enabled"] = context.get("sentiment", {}).get("enabled", False)

        with open(f"{ART_DIR}/decision.json", "w") as f:
            json.dump(decision_data, f, indent=2)

    # ✅ Enhanced trace append with more context
    if context.get("explainability", {}).get("full_trace"):
        with open(f"{ART_DIR}/trace.jsonl", "a") as f:
            trace_entry = {
                "timestamp": datetime.utcnow().isoformat() + "Z",
                "symbol": context["data"].get("symbol", "BTC"),
                "actor": "Pipeline",
                "trace_step": context["explainability"]["full_trace"][-1],
                "forecast_model": context.get("forecast", {}).get("model_used"),
                "sentiment_enabled": context.get("sentiment", {}).get("enabled", False)
            }
            f.write(json.dumps(trace_entry) + "\n")

    # ✅ Enhanced correlation log CSV with ALL advanced sentiment metrics
    try:
        hist = context.get("history", [])
        if hist:
            path = f"{ART_DIR}/correlation_log.csv"
            file_exists = os.path.exists(path)

            # ✅ ENHANCED FIELDNAMES with all new sentiment metrics
            fieldnames = [
                # Basic info
                "timestamp", "symbol",
                # Forecast metrics
                "forecast_movement", "forecast_confidence", "forecast_model",
                # Enhanced sentiment metrics
                "sentiment_verdict", "sentiment_score", "sentiment_confidence",
                "sentiment_strength_category", "news_volume", "recency_score",
                "sentiment_directional_accuracy",
                # Decision metrics
                "alignment", "used_sentiment", "decision_action"
            ]

            with open(path, "a", newline="") as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()

                # Write only the last row from history with all enhanced metrics
                last = hist[-1]
                sentiment = context.get("sentiment", {})
                forecast = context.get("forecast", {})
                decision = context.get("decision", {})

                writer.writerow({
                    # Basic info
                    "timestamp": context["data"].get("timestamp"),
                    "symbol": context["data"].get("symbol", "BTC"),

                    # Forecast metrics
                    "forecast_movement": last.get("forecast_movement"),
                    "forecast_confidence": last.get("forecast_confidence"),
                    "forecast_model": forecast.get("model_used"),

                    # Enhanced sentiment metrics
                    "sentiment_verdict": last.get("sentiment_verdict"),
                    "sentiment_score": last.get("sentiment_score"),
                    "sentiment_confidence": last.get("sentiment_confidence"),
                    "sentiment_strength_category": sentiment.get("strength_category"),
                    "news_volume": sentiment.get("news_volume"),
                    "recency_score": sentiment.get("recency_score"),
                    "sentiment_directional_accuracy": sentiment.get("directional_accuracy"),

                    # Decision metrics
                    "alignment": last.get("alignment"),
                    "used_sentiment": last.get("used_sentiment"),
                    "decision_action": decision.get("action")
                })

    except Exception as e:
        # Non-fatal; continue pipeline
        print(f"Warning: CSV persistence failed: {e}")
        pass

    # ✅ NEW: Save model performance metrics for analysis
    try:
        if context.get("forecast", {}).get("metrics_per_model"):
            metrics_data = {
                "timestamp": context["data"].get("timestamp"),
                "symbol": context["data"].get("symbol"),
                "selected_model": context["forecast"].get("model_used"),
                "metrics": context["forecast"]["metrics_per_model"]
            }

            with open(f"{ART_DIR}/model_performance.json", "w") as f:
                json.dump(metrics_data, f, indent=2)

    except Exception as e:
        # Non-fatal
        pass

class OrchestratorChain:
    """
    Enhanced practical MCP orchestrator for crypto trading pipeline
    Executes agents sequentially with comprehensive error handling and persistence
    """

    def __init__(self, agents, persist=None):
        self.agents = agents  # [DataFetcherAgent(), ForecastAgent(), SentimentAgent(), DecisionAgent()]
        self.persist = persist

    def run(self, context):
        """
        Execute the complete agent pipeline with enhanced error handling
        """
        successful_agents = 0
        total_agents = len(self.agents)

        context["explainability"]["full_trace"].append(
            f"[OrchestratorChain] Starting pipeline with {total_agents} agents"
        )

        for i, agent in enumerate(self.agents, 1):
            agent_name = agent.__class__.__name__

            try:
                # Execute agent
                context = agent.run(context)
                successful_agents += 1

                # Log success
                if context.get("explainability", {}).get("enabled"):
                    context["explainability"]["full_trace"].append(
                        f"[OrchestratorChain] ✅ {agent_name} completed ({i}/{total_agents})"
                    )

                # Persist after each successful agent (for incremental backup)
                if self.persist:
                    self.persist(context)

            except Exception as e:
                # Log failure but continue pipeline where possible
                error_msg = f"[OrchestratorChain] ❌ {agent_name} failed: {e}"
                context["explainability"]["full_trace"].append(error_msg)

                # For critical agents, might want to stop pipeline
                if agent_name in ["DataFetcherAgent", "ForecastAgent"]:
                    context["explainability"]["full_trace"].append(
                        f"[OrchestratorChain] 🛑 Critical agent {agent_name} failed - stopping pipeline"
                    )
                    break
                else:
                    # Non-critical agents can fail gracefully
                    context["explainability"]["full_trace"].append(
                        f"[OrchestratorChain] ⚠️ Non-critical agent {agent_name} failed - continuing pipeline"
                    )

        # Final pipeline summary
        context["explainability"]["full_trace"].append(
            f"[OrchestratorChain] Pipeline completed: {successful_agents}/{total_agents} agents successful"
        )

        # Final log and persistence
        try:
            context = finalize_and_log(context)
            if self.persist:
                self.persist(context)  # Final save
        except Exception as e:
            context["explainability"]["full_trace"].append(
                f"[OrchestratorChain] ⚠️ Final logging failed: {e}"
            )

        return context

# ✅ Enhanced convenience function for quick testing
def run_enhanced_pipeline(symbol="BTC", use_sentiment=True, save_artifacts=True):
    """
    Convenience function to run the complete enhanced pipeline
    Perfect for testing and development
    """

    # Fresh context
    test_context = context.copy()
    test_context["data"]["symbol"] = symbol
    test_context["sentiment"]["enabled"] = use_sentiment

    # Initialize agents
    agents = [
        DataFetcherAgent(),
        ForecastAgent(),
        SentimentAgent(),
        DecisionAgent()
    ]

    # Initialize orchestrator with optional persistence
    orchestrator = OrchestratorChain(
        agents=agents,
        persist=persist_artifacts if save_artifacts else None
    )

    # Execute pipeline
    print(f"🚀 Running enhanced pipeline for {symbol} (Sentiment: {use_sentiment})")
    print("-" * 60)

    result = orchestrator.run(test_context)

    # Quick summary
    decision = result.get("decision", {})
    forecast = result.get("forecast", {})
    sentiment = result.get("sentiment", {})

    print(f"✅ Pipeline completed!")
    print(f"📊 Model: {forecast.get('model_used', 'N/A')}")
    print(f"📈 Prediction: {forecast.get('predicted_movement', 'N/A')} ({forecast.get('confidence', 0):.1%})")

    if sentiment.get("enabled"):
        print(f"🧠 Sentiment: {sentiment.get('verdict', 'N/A')} ({sentiment.get('strength_category', 'N/A')} strength)")
        print(f"📰 News: {sentiment.get('news_volume', 0)} articles, {sentiment.get('recency_score', 0):.1%} fresh")

    print(f"💡 Decision: {decision.get('action', 'N/A')}")

    if save_artifacts:
        print(f"💾 Artifacts saved to {ART_DIR}/ folder")

    return result

In [7]:

# Cell 7 – Enhanced Run the Agentic Chain with Advanced Sentiment Metrics
# This cell creates a fresh context and uses the enhanced OrchestratorChain from Cell 6 to execute
# the full pipeline with modular agents, displaying all advanced sentiment and forecast metrics.
# Perfect for testing the complete enhanced system end-to-end.

# === Enhanced Agentic Chain Execution with Advanced Metrics Display ===
from datetime import datetime
from tabulate import tabulate

print("🚀 TESTING ENHANCED AGENTIC ORCHESTRATOR CHAIN")
print("=" * 70)

# Fresh context (enhanced with all new fields)
final_chain_ctx = None
chain_context = {
    "data": {
        "symbol": "BTC",
        "timestamp": datetime.today().strftime("%Y-%m-%d"),
    },
    "forecast": {},
    "sentiment": {"enabled": True},
    "decision": {},
    "explainability": {"enabled": True, "full_trace": []},
    "history": [],
    "sentiment_history": []  # ✅ Enhanced with sentiment tracking
}

# Build and run enhanced chain
chain = OrchestratorChain(
    agents=[DataFetcherAgent(), ForecastAgent(), SentimentAgent(), DecisionAgent()],
    persist=persist_artifacts
)

print("⏳ Executing enhanced pipeline...")
final_chain_ctx = chain.run(chain_context)
print("✅ Pipeline completed!\n")

# === ENHANCED SUMMARY DISPLAY ===

# 1. Final Decision
print("══════════════════════════════════════════════════")
print("📈 FINAL TRADING DECISION (Enhanced Agentic Chain)")
print("══════════════════════════════════════════════════")
print(f"🔹 Action: {final_chain_ctx['decision'].get('action', 'N/A').upper()}")
print(f"🔹 Rationale: {final_chain_ctx['decision'].get('rationale', 'N/A')}\n")

# 2. Enhanced Forecast Details
print("🔍 ENHANCED FORECAST DETAILS")
print(f"🔹 Model Used: {final_chain_ctx['forecast'].get('model_used', 'N/A')} (selected by composite score)")
print(f"🔹 Predicted Movement: {final_chain_ctx['forecast'].get('predicted_movement', 'N/A').upper()}")
print(f"🔹 Confidence Score: {final_chain_ctx['forecast'].get('confidence', 0):.1%}")

mpm = final_chain_ctx["forecast"].get("metrics_per_model", {})
bm = final_chain_ctx['forecast'].get('model_used')
if bm and bm in mpm:
    best_metrics = mpm[bm]
    print(f"🎯 Key Performance Metrics:")
    print(f"   • Directional Accuracy: {best_metrics.get('Directional_Accuracy', 0):.1%} (most important for trading)")
    print(f"   • MAPE: {best_metrics.get('MAPE', 0):.1f}% (scale-independent error)")
    print(f"   • MAE: ${best_metrics.get('MAE', 0):.2f} (absolute error magnitude)")
    print(f"   • RMSE: ${best_metrics.get('RMSE', 0):.2f} (reference metric)")
print()

# 3. ✅ ENHANCED SENTIMENT ANALYSIS DISPLAY
if final_chain_ctx["sentiment"].get("enabled"):
    print("🧠 ENHANCED SENTIMENT ANALYSIS")
    print(f"🔹 Verdict: {final_chain_ctx['sentiment'].get('verdict', 'N/A').upper()} ({final_chain_ctx['sentiment'].get('strength_category', 'N/A')} Strength)")
    print(f"🔹 Score: {final_chain_ctx['sentiment'].get('score', 0):.3f} (range: -1 to +1)")

    # ✅ NEW ADVANCED METRICS DISPLAY
    print(f"📊 Sentiment Performance Metrics:")
    print(f"   • News Volume: {final_chain_ctx['sentiment'].get('news_volume', 0)} articles analyzed")
    print(f"   • News Freshness: {final_chain_ctx['sentiment'].get('recency_score', 0):.1%} (recent news weight)")
    print(f"   • Historical Accuracy: {final_chain_ctx['sentiment'].get('directional_accuracy', 0):.1%} (directional hit rate)")
    print(f"   • Enhanced Confidence: {final_chain_ctx['sentiment'].get('confidence', 0):.1%} (volume + agreement + recency)")

    # Alignment display
    alignment_status = final_chain_ctx['sentiment'].get('alignment')
    if alignment_status is True:
        print(f"🔹 Forecast-Sentiment Alignment: ✅ ALIGNED (threshold boost)")
    elif alignment_status is False:
        print(f"🔹 Forecast-Sentiment Alignment: ❌ CONFLICTING (threshold penalty)")
    else:
        print(f"🔹 Forecast-Sentiment Alignment: ➖ NEUTRAL (no adjustment)")

    print(f"🔹 Used in Decision: {'✅ YES' if final_chain_ctx['sentiment'].get('used_in_decision') else '❌ NO'}")

    # ✅ SAMPLE HEADLINES FOR TRANSPARENCY
    headlines = final_chain_ctx['sentiment'].get('headlines_sample', [])
    if headlines:
        print(f"📰 Sample Headlines Analyzed:")
        for i, headline in enumerate(headlines, 1):
            print(f"   {i}. {headline[:80]}{'...' if len(headline) > 80 else ''}")
    print()

# 4. Explainability Trace
print("🛡️ EXPLAINABILITY TRACE")
for i, step in enumerate(final_chain_ctx["explainability"]["full_trace"], 1):
    print(f"{i:02d}. {step}")
print()

# 5. ✅ ENHANCED FORECAST METRICS TABLE with Directional Accuracy
if mpm:
    print("📊 ENHANCED MODEL PERFORMANCE COMPARISON")
    print("🎯 Ranked by Composite Score: 40% Directional Accuracy + 30% MAPE + 30% MAE")

    model_table = []
    selected_model = final_chain_ctx["forecast"].get("model_used")

    # Calculate composite scores for ranking
    model_scores = []
    for model_name, metrics in mpm.items():
        # Recreate composite score calculation
        latest_price = 50000  # Approximate, for display purposes
        dir_acc_score = metrics.get("Directional_Accuracy", 0)
        mape_score = max(0, 1 - metrics.get("MAPE", 0) / 100)
        normalized_mae = metrics.get("MAE", 0) / latest_price
        mae_score = max(0, 1 - normalized_mae)
        composite = 0.4 * dir_acc_score + 0.3 * mape_score + 0.3 * mae_score

        model_scores.append((model_name, composite, metrics))

    # Sort by composite score (descending)
    model_scores.sort(key=lambda x: x[1], reverse=True)

    for rank, (model_name, composite_score, metrics) in enumerate(model_scores, 1):
        selected_indicator = "👑" if model_name == selected_model else f"{rank}."
        model_table.append([
            f"{selected_indicator} {model_name}",
            f"{metrics.get('Directional_Accuracy', 0):.1%}",
            f"{metrics.get('MAPE', 0):.1f}%",
            f"${metrics.get('MAE', 0):.2f}",
            f"${metrics.get('RMSE', 0):.2f}",
            f"{composite_score:.3f}"
        ])

    print(tabulate(
        model_table,
        headers=["Model", "Dir Acc ↑", "MAPE ↓", "MAE ↓", "RMSE ↓", "Score ↑"],
        tablefmt="fancy_grid"
    ))
    print("Legend: ↑ = Higher is better, ↓ = Lower is better, 👑 = Selected model")
    print()

# ✅ 6. NEW SENTIMENT PERFORMANCE BREAKDOWN TABLE
if final_chain_ctx["sentiment"].get("enabled"):
    print("🧠 SENTIMENT ANALYSIS BREAKDOWN")

    sentiment_performance = [
        ["Verdict", final_chain_ctx['sentiment'].get('verdict', 'N/A').upper()],
        ["Strength Category", final_chain_ctx['sentiment'].get('strength_category', 'N/A')],
        ["Polarity Score", f"{final_chain_ctx['sentiment'].get('score', 0):.3f}"],
        ["News Volume", f"{final_chain_ctx['sentiment'].get('news_volume', 0)} articles"],
        ["News Freshness", f"{final_chain_ctx['sentiment'].get('recency_score', 0):.1%}"],
        ["Historical Accuracy", f"{final_chain_ctx['sentiment'].get('directional_accuracy', 0):.1%}"],
        ["Enhanced Confidence", f"{final_chain_ctx['sentiment'].get('confidence', 0):.1%}"]
    ]

    print(tabulate(
        sentiment_performance,
        headers=["Metric", "Value"],
        tablefmt="fancy_grid"
    ))
    print()

# ✅ 7. ENHANCED CORRELATION HISTORY with All New Metrics
hist = final_chain_ctx.get("history", [])
if hist:
    print("📈 COMPREHENSIVE DECISION ANALYSIS HISTORY")
    corr_table = []

    for h in hist:
        # Skip any entries missing required fields
        required_keys = ["timestamp", "forecast_movement", "forecast_confidence", "sentiment_verdict"]
        if not all(k in h for k in required_keys):
            continue

        # Format alignment with symbols
        alignment_symbol = "✅" if h.get("alignment") is True else "❌" if h.get("alignment") is False else "➖"

        corr_table.append([
            h["timestamp"],
            h["forecast_movement"].upper(),
            f"{h['forecast_confidence']:.1%}",
            h["sentiment_verdict"].upper(),
            h.get("sentiment_strength", "N/A"),
            h.get("news_volume", "N/A"),
            f"{h.get('recency_score', 0):.1%}",
            alignment_symbol,
            "✅" if h.get("used_sentiment") else "❌"
        ])

    if corr_table:
        print(tabulate(
            corr_table,
            headers=["Date", "Forecast", "F_Conf", "Sentiment", "Strength", "Volume", "Fresh", "Align", "Used"],
            tablefmt="fancy_grid"
        ))
        print("Legend: Align: ✅=Aligned, ❌=Conflicting, ➖=Neutral | Fresh=News Freshness")
    else:
        print("No valid correlation entries available.")
else:
    print("📈 No correlation history available yet.")

# ✅ 8. ARTIFACTS SUMMARY
print(f"\n💾 ARTIFACTS PERSISTENCE")
print(f"📁 Location: {ART_DIR}/ folder")
print(f"📄 Files created:")
print(f"   • forecast_results.json (forecast metrics & model selection)")
print(f"   • sentiment_results.json (all advanced sentiment metrics)")
print(f"   • decision.json (trading decision & rationale)")
print(f"   • correlation_log.csv (historical analysis data)")
print(f"   • trace.jsonl (explainability trace log)")
print(f"   • model_performance.json (model comparison data)")

print("\n" + "="*80)
print("🎓 ENHANCED MCP-TRADI-WIN: Complete Agentic Pipeline Test")
print("   ✅ Advanced Sentiment Metrics: Volume + Freshness + Historical Accuracy")
print("   ✅ Enhanced Forecast Metrics: Directional Accuracy + MAPE + MAE")
print("   ✅ Comprehensive Persistence: JSON + CSV + JSONL artifacts")
print("   ✅ Full Explainability: Step-by-step trace with agent coordination")
print("="*80)

# ✅ 9. QUICK ACCESS TO RESULTS
print(f"\n🔗 Quick Access Variables:")
print(f"   final_chain_ctx = Complete results context")
print(f"   final_chain_ctx['decision']['action'] = '{final_chain_ctx.get('decision', {}).get('action', 'N/A')}'")
print(f"   final_chain_ctx['forecast']['model_used'] = '{final_chain_ctx.get('forecast', {}).get('model_used', 'N/A')}'")
print(f"   final_chain_ctx['sentiment']['verdict'] = '{final_chain_ctx.get('sentiment', {}).get('verdict', 'N/A')}'")

🚀 TESTING ENHANCED AGENTIC ORCHESTRATOR CHAIN
⏳ Executing enhanced pipeline...


INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: /tmp/tmpiog_xh3b/5xo6rte1.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpiog_xh3b/rjdbcg2q.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.11/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=55506', 'data', 'file=/tmp/tmpiog_xh3b/5xo6rte1.json', 'init=/tmp/tmpiog_xh3b/rjdbcg2q.json', 'output', 'file=/tmp/tmpiog_xh3b/prophet_model0pa_0phr/prophet_model-20250815035903.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
03:59:03 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
03:59:03 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


✅ Latest Fetched Date: 2025-08-15
✅ Pipeline completed!

══════════════════════════════════════════════════
📈 FINAL TRADING DECISION (Enhanced Agentic Chain)
══════════════════════════════════════════════════
🔹 Action: SELL
🔹 Rationale: Forecast DOWN with confidence 0.925 (threshold 0.60); sentiment=neutral.

🔍 ENHANCED FORECAST DETAILS
🔹 Model Used: Prophet (selected by composite score)
🔹 Predicted Movement: DOWN
🔹 Confidence Score: 92.5%
🎯 Key Performance Metrics:
   • Directional Accuracy: 88.9% (most important for trading)
   • MAPE: 1.6% (scale-independent error)
   • MAE: $1913.54 (absolute error magnitude)
   • RMSE: $2347.33 (reference metric)

🧠 ENHANCED SENTIMENT ANALYSIS
🔹 Verdict: NEUTRAL (Weak Strength)
🔹 Score: 0.053 (range: -1 to +1)
📊 Sentiment Performance Metrics:
   • News Volume: 10 articles analyzed
   • News Freshness: 50.0% (recent news weight)
   • Historical Accuracy: 65.0% (directional hit rate)
   • Enhanced Confidence: 56.1% (volume + agreement + recency)
🔹 F

In [8]:
#Cell 8
import os
os.makedirs("tradiwin_core", exist_ok=True)

In [9]:
#Cell 9
%%writefile tradiwin_core/__init__.py
# Re-export the public API so Streamlit can `from tradiwin_core import ...`
from .core import (
    OrchestratorChain,
    MCP_tradi_win_runner,
    build_default_context,
    load_prices,
    mini_backtest,
)

Writing tradiwin_core/__init__.py


In [17]:
# Cell 10 — FIXED tradiwin_core/core.py with CORRECTED Directional Accuracy
%%writefile tradiwin_core/core.py
from __future__ import annotations
import os, warnings, random
from typing import Dict, Any, Tuple
import numpy as np
import pandas as pd
import yfinance as yf
from statsmodels.tsa.arima.model import ARIMA
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

def build_default_context(symbol: str = "BTC") -> Dict[str, Any]:
    return {
        "data": {"symbol": symbol, "df": None, "timestamp": None},
        "forecast": {"model_used": None, "predicted_movement": None, "confidence": 0.0,
                     "metrics_per_model": {}, "skipped": [], "forecast_value": None, "latest_price": None},
        "sentiment": {
            "enabled": False, "source": None, "score": 0.0, "verdict": "neutral",
            "confidence": 0.0, "used_in_decision": False, "alignment": None, "sample_headlines": [],
            # ✅ Enhanced sentiment metrics
            "strength_category": None, "news_volume": None, "recency_score": None,
            "directional_accuracy": None
        },
        "decision": {"action": None, "rationale": None, "score": 0.0},
        "history": [],
        "sentiment_history": [],  # ✅ For sentiment directional accuracy tracking
        "explainability": {"enabled": True, "full_trace": []},
        "debug_info": {"enabled": False, "directional_accuracy_details": []}  # ✅ Debug mode
    }

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = ["_".join([str(x) for x in tup if str(x) != ""]).strip()
                      for tup in df.columns.values]
    return df

def load_prices(symbol: str, days: int = 365) -> pd.DataFrame:
    df = yf.download(tickers=f"{symbol}-USD", period=f"{days}d", interval="1d", progress=False)
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["Date", "Close"])
    df = df.reset_index(drop=False)
    df = _flatten_columns(df)

    # normalize Date
    date_col = None
    for c in df.columns:
        if c == "Date":
            date_col = c; break
    if date_col is None:
        for c in df.columns:
            lc = c.lower()
            if lc in ("datetime","timestamp") or "date" in lc:
                date_col = c; break
    if date_col is None:
        first = df.columns[0]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            probe = pd.to_datetime(df[first], errors="coerce")
        if probe.notna().any():
            df.insert(0, "Date", probe)
            date_col = "Date"
    if date_col != "Date":
        df.rename(columns={date_col: "Date"}, inplace=True)
    if "Date" not in df.columns:
        return pd.DataFrame(columns=["Date", "Close"])

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df[df["Date"].notna()]
    df["Date"] = df["Date"].dt.tz_localize(None).dt.normalize()

    # normalize Close
    close_col = None
    for c in df.columns:
        if c.lower().startswith("close"):
            close_col = c; break
    if close_col is None:
        for c in df.columns:
            if "close" in c.lower():
                close_col = c; break
    if close_col is not None and close_col != "Close":
        df.rename(columns={close_col: "Close"}, inplace=True)
    if "Close" not in df.columns:
        return pd.DataFrame(columns=["Date", "Close"])

    df = df[df["Close"].notna()]
    df = df.sort_values("Date").drop_duplicates(subset=["Date"], keep="last")
    return df[["Date", "Close"]]

def compute_metrics(y_true, y_pred) -> Tuple[float, float, float]:
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    rmse = float(np.sqrt(((y_true - y_pred) ** 2).mean()))
    mae  = float(np.abs(y_true - y_pred).mean())
    eps = 1e-8; den = np.maximum(np.abs(y_true), eps)
    mape = float(np.mean(np.abs((y_true - y_pred) / den)) * 100.0)
    return rmse, mae, mape

# ✅ GREATLY IMPROVED: Enhanced Directional Accuracy Calculation
def calculate_directional_accuracy(y_true, y_pred, min_change_pct=0.1, debug_context=None) -> float:
    """
    GREATLY IMPROVED directional accuracy calculation with much better noise filtering

    Args:
        y_true: Actual prices
        y_pred: Predicted prices
        min_change_pct: Minimum percentage change to consider (default 0.1% - much lower!)
        debug_context: Context for debug logging

    Returns:
        Directional accuracy as float (0.0 to 1.0)
    """
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)

    # Ensure we have enough data
    if len(y_true) < 2 or len(y_pred) < 2:
        if debug_context:
            debug_context["debug_info"]["directional_accuracy_details"].append({
                "error": "Insufficient data",
                "y_true_length": len(y_true),
                "y_pred_length": len(y_pred)
            })
        return 0.0

    # Ensure arrays are same length
    min_len = min(len(y_true), len(y_pred))
    y_true = y_true[:min_len]
    y_pred = y_pred[:min_len]

    if min_len < 2:
        return 0.0

    # GREATLY IMPROVED: Calculate percentage changes with much better precision
    actual_changes = np.diff(y_true) / y_true[:-1] * 100  # Percentage changes
    predicted_changes = np.diff(y_pred) / y_pred[:-1] * 100  # Percentage changes

    # GREATLY IMPROVED: Much lower noise threshold for better signal detection (0.1% instead of 0.3%)
    significant_moves = np.abs(actual_changes) >= min_change_pct

    if not significant_moves.any():
        # If no significant moves, return neutral accuracy
        if debug_context:
            debug_context["debug_info"]["directional_accuracy_details"].append({
                "warning": "No significant price movements found",
                "min_change_threshold": f"{min_change_pct}%",
                "max_actual_change": f"{np.abs(actual_changes).max():.2f}%"
            })
        return 0.5  # Neutral - no significant movements to predict

    # Only consider significant movements
    filtered_actual = actual_changes[significant_moves]
    filtered_predicted = predicted_changes[significant_moves]

    # Determine directions (1 for up, -1 for down, 0 for flat)
    actual_directions = np.sign(filtered_actual)
    predicted_directions = np.sign(filtered_predicted)

    # Calculate accuracy
    correct_predictions = (actual_directions == predicted_directions).sum()
    total_predictions = len(filtered_actual)

    if total_predictions == 0:
        return 0.5

    accuracy = correct_predictions / total_predictions

    # ✅ Enhanced debug logging
    if debug_context:
        debug_context["debug_info"]["directional_accuracy_details"].append({
            "total_periods": min_len - 1,
            "significant_moves": int(significant_moves.sum()),
            "correct_predictions": int(correct_predictions),
            "total_predictions": total_predictions,
            "accuracy": f"{accuracy:.1%}",
            "min_change_threshold": f"{min_change_pct}%",
            "sample_actual_changes": [f"{x:.2f}%" for x in actual_changes[:5]],
            "sample_predicted_changes": [f"{x:.2f}%" for x in predicted_changes[:5]],
            "sample_directions_actual": actual_directions[:5].tolist(),
            "sample_directions_predicted": predicted_directions[:5].tolist()
        })

    return float(accuracy)

# ✅ ENHANCED: Composite Score for Model Selection
def composite_score(metrics: Dict[str, float], latest_price: float) -> float:
    """Combine Directional Accuracy (40%), MAPE (30%), and MAE (30%) for model selection"""
    try:
        normalized_mae = metrics["MAE"] / max(latest_price, 1e-8)

        # Convert to "goodness" scores (higher = better)
        dir_acc_score = metrics["Directional_Accuracy"]
        mape_score = max(0, 1 - metrics["MAPE"] / 100)
        mae_score = max(0, 1 - normalized_mae)

        # Weighted combination: 40% Dir Acc, 30% MAPE, 30% MAE
        composite = 0.4 * dir_acc_score + 0.3 * mape_score + 0.3 * mae_score
        return composite
    except:
        return 0.0

def _guard_ok(forecast_value: float, latest: float, rmse: float) -> bool:
    if not (np.isfinite(forecast_value) and np.isfinite(latest) and np.isfinite(rmse)): return False
    if latest <= 0: return False
    if rmse / latest > 0.80: return False
    if abs(forecast_value - latest) / latest > 0.50: return False
    return True

def _news_query(symbol: str) -> str:
    return {"BTC":"bitcoin","ETH":"ethereum","SOL":"solana","XRP":"ripple xrp","ADA":"cardano","BNB":"binance coin"}.get(symbol.upper(), symbol)

def mini_backtest(close_df: pd.DataFrame, asof_ts: pd.Timestamp, lookback_days: int = 30) -> pd.DataFrame:
    """
    FIXED mini backtest with corrected typo and enhanced directional accuracy calculation
    """
    df = close_df.copy()
    if df.empty: return pd.DataFrame()
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None).dt.normalize()
    asof_ts = pd.Timestamp(asof_ts).normalize()
    df = df[df["Date"] <= asof_ts].copy()
    if len(df) < 60: return pd.DataFrame()

    # FIXED: Corrected typo from "lookbook_days" to "lookback_days"
    tail = df.tail(lookback_days + 1).reset_index(drop=True)
    rows = []
    all_predictions = []
    all_actuals = []

    # Build all predictions first for consistent directional accuracy
    for i in range(len(tail) - 1):
        cut_date = pd.Timestamp(tail["Date"].iloc[i]).normalize()
        hist = df[df["Date"] <= cut_date]["Close"].to_numpy()
        try:
            f = float(ARIMA(hist, order=(3,1,0)).fit().forecast()[0])
        except Exception:
            f = float(hist[-1])

        actual_next = float(tail.loc[i+1, "Close"])
        all_predictions.append(f)
        all_actuals.append(actual_next)

    # GREATLY IMPROVED: Use enhanced directional accuracy with much lower threshold
    overall_dir_acc = calculate_directional_accuracy(all_actuals, all_predictions, min_change_pct=0.1)

    # Build individual rows
    for i in range(len(tail) - 1):
        cut_date = pd.Timestamp(tail["Date"].iloc[i]).normalize()
        hist = df[df["Date"] <= cut_date]["Close"].to_numpy()

        try:
            f = float(ARIMA(hist, order=(3,1,0)).fit().forecast()[0])
        except Exception:
            f = float(hist[-1])

        next_day = pd.Timestamp(tail["Date"].iloc[i+1]).normalize()
        actual_next = float(tail.loc[i+1, "Close"])
        latest = float(hist[-1])

        # IMPROVED: Use consistent confidence calculation with forecast agent
        price_error_pct = abs(f - actual_next) / actual_next
        rmse_conf = max(0.0, min(1.0, 1 - price_error_pct))
        confidence = 0.6 * overall_dir_acc + 0.4 * rmse_conf  # Same as forecast confidence

        # Enhanced directional accuracy for individual prediction
        pred_dir = "UP" if f > latest else "DOWN" if f < latest else "STABLE"
        actual_dir = "UP" if actual_next > latest else "DOWN" if actual_next < latest else "STABLE"
        dir_correct = (pred_dir == actual_dir)

        rows.append({
            "date": next_day.date(),
            "predicted_close": f,
            "actual_close": actual_next,
            "pred_direction": pred_dir,
            "actual_direction": actual_dir,
            "directional_correct": dir_correct,
            "conf": confidence  # IMPROVED: Now consistent with other confidence calculations
        })

    return pd.DataFrame(rows)

class AgentBase:
    def run(self, context: Dict[str, Any]) -> Dict[str, Any]:
        raise NotImplementedError

class DataFetcherAgent(AgentBase):
    def __init__(self, days: int = 365): self.days = days
    def run(self, context: Dict[str, Any]) -> Dict[str, Any]:
        symbol = context["data"]["symbol"]
        df = load_prices(symbol, days=self.days)
        asof_str = context["data"].get("timestamp")
        if asof_str:
            asof_ts = pd.to_datetime(asof_str).tz_localize(None).normalize()
            df = df[df["Date"] <= asof_ts].copy()
        if df.empty:
            context["explainability"]["full_trace"].append("[Data] No price data.")
            context["data"]["df"] = pd.DataFrame(columns=["Date","Close"])
            context["data"]["timestamp"] = None
            return context
        context["data"]["df"] = df
        context["data"]["timestamp"] = str(df["Date"].iloc[-1].date())
        context["explainability"]["full_trace"].append(
            f"[Data] Loaded {len(df)} rows | range {df['Date'].min().date()} → {df['Date'].max().date()}"
        )
        return context

class ForecastAgent(AgentBase):
    def run(self, context: Dict[str, Any]) -> Dict[str, Any]:
        df = context["data"].get("df")
        if df is None or df.empty:
            context["explainability"]["full_trace"].append("[Forecast] No data.")
            return context
        prices = df["Close"].to_numpy(dtype=float)
        if len(prices) < 30:
            context["explainability"]["full_trace"].append("[Forecast] Not enough history (<30).")
            return context
        latest = float(prices[-1])
        results, metrics, skipped = {}, {}, []

        # ✅ GREATLY IMPROVED ARIMA with Enhanced Directional Accuracy
        try:
            arima = ARIMA(prices, order=(3,1,0)).fit()
            f = float(arima.forecast()[0])
            pred = arima.predict()

            # IMPROVED: Use larger validation window for better accuracy
            validation_size = min(35, len(pred))  # Increased from 30 to 35
            rmse, mae, mape = compute_metrics(prices[-validation_size:], pred[-validation_size:])

            # ✅ GREATLY IMPROVED directional accuracy with much lower threshold
            dir_acc = calculate_directional_accuracy(
                prices[-validation_size:],
                pred[-validation_size:],
                min_change_pct=0.1,  # Much lower threshold for better detection
                debug_context=context
            )

            if _guard_ok(f, latest, rmse):
                results["ARIMA"] = f
                metrics["ARIMA"] = {"RMSE": rmse, "MAE": mae, "MAPE": mape, "Directional_Accuracy": dir_acc}
            else:
                skipped.append(("ARIMA", f"guardrails (RMSE={rmse:.0f})"))
        except Exception as e:
            skipped.append(("ARIMA", f"error: {str(e).splitlines()[0][:120]}"))

        # ✅ GREATLY IMPROVED PROPHET with Enhanced Directional Accuracy
        try:
            from prophet import Prophet
            d = df.rename(columns={"Date":"ds","Close":"y"})[["ds","y"]].copy()
            if len(d) < 180: m = Prophet(weekly_seasonality=False, yearly_seasonality=False)
            else:            m = Prophet()
            m.fit(d)
            fut = m.make_future_dataframe(periods=1, freq="D")
            fc = m.predict(fut)
            f = float(fc["yhat"].iloc[-1])

            # IMPROVED: Use larger validation window
            validation_size = min(35, len(fc) - 1)  # Increased from 30 to 35
            y_true = d["y"].iloc[-validation_size:].to_numpy()
            y_pred = fc["yhat"].iloc[-(validation_size+1):-1].to_numpy()

            rmse, mae, mape = compute_metrics(y_true, y_pred)

            # ✅ GREATLY IMPROVED directional accuracy
            dir_acc = calculate_directional_accuracy(
                y_true,
                y_pred,
                min_change_pct=0.1,  # Much lower threshold
                debug_context=context
            )

            if _guard_ok(f, latest, rmse):
                results["Prophet"] = f
                metrics["Prophet"] = {"RMSE": rmse, "MAE": mae, "MAPE": mape, "Directional_Accuracy": dir_acc}
            else:
                skipped.append(("Prophet", f"guardrails (RMSE={rmse:.0f})"))
        except Exception as e:
            skipped.append(("Prophet", f"error: {str(e).splitlines()[0][:120]}"))

        # ✅ GREATLY IMPROVED LSTM with Enhanced Directional Accuracy
        try:
            import tensorflow as tf
            from tensorflow.keras.models import Sequential
            from tensorflow.keras.layers import LSTM, Dense, Dropout
            from sklearn.preprocessing import MinMaxScaler
            np.random.seed(42); random.seed(42)
            try: tf.random.set_seed(42)
            except Exception: pass

            if len(prices) >= 10:
                # Enhanced LSTM with returns-based prediction
                returns = np.diff(prices) / prices[:-1] * 100
                sc = MinMaxScaler(feature_range=(-1, 1))
                scaled_returns = sc.fit_transform(returns.reshape(-1, 1)).flatten()

                lookback = min(12, len(scaled_returns) - 1)
                X, y = [], []
                for i in range(len(scaled_returns) - lookback):
                    X.append(scaled_returns[i:i+lookback])
                    y.append(scaled_returns[i+lookback])
                X, y = np.array(X), np.array(y)

                if len(X) > 0:
                    # Enhanced architecture with dropout
                    model = Sequential([
                        LSTM(50, return_sequences=True, input_shape=(lookback, 1)),
                        Dropout(0.2),
                        LSTM(25, return_sequences=False),
                        Dropout(0.2),
                        Dense(1)
                    ])
                    model.compile(optimizer="adam", loss="mse")
                    model.fit(X.reshape(X.shape[0], X.shape[1], 1), y, epochs=20, verbose=0, validation_split=0.2)

                    # Predict next return
                    last_sequence = scaled_returns[-lookback:].reshape(1, lookback, 1)
                    predicted_return_scaled = model.predict(last_sequence, verbose=0)[0][0]
                    predicted_return = sc.inverse_transform([[predicted_return_scaled]])[0][0]
                    f = float(latest * (1 + predicted_return / 100))

                    # IMPROVED: Calculate metrics on larger validation set
                    validation_size = min(30, len(X))  # Increased validation size
                    y_pred_scaled = model.predict(X.reshape(X.shape[0], X.shape[1], 1)[-validation_size:], verbose=0).flatten()
                    y_pred_returns = sc.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
                    y_true_returns = sc.inverse_transform(y[-validation_size:].reshape(-1, 1)).flatten()

                    # Convert back to prices
                    base_prices = prices[-(validation_size+1):-1]
                    y_pred_prices = base_prices * (1 + y_pred_returns / 100)
                    y_true_prices = base_prices * (1 + y_true_returns / 100)

                    rmse, mae, mape = compute_metrics(y_true_prices, y_pred_prices)

                    # ✅ GREATLY IMPROVED directional accuracy
                    dir_acc = calculate_directional_accuracy(
                        y_true_prices,
                        y_pred_prices,
                        min_change_pct=0.1,  # Much lower threshold
                        debug_context=context
                    )

                    if _guard_ok(f, latest, rmse):
                        results["LSTM"] = f
                        metrics["LSTM"] = {"RMSE": rmse, "MAE": mae, "MAPE": mape, "Directional_Accuracy": dir_acc}
                    else:
                        skipped.append(("LSTM", f"guardrails (RMSE={rmse:.0f})"))
                else:
                    skipped.append(("LSTM","insufficient sequence"))
            else:
                skipped.append(("LSTM","not enough rows"))
        except Exception as e:
            skipped.append(("LSTM", f"error: {str(e).splitlines()[0][:120]}"))

        if not metrics:
            context["forecast"].update({"model_used":"none","predicted_movement":"stable","confidence":0.0,
                                        "metrics_per_model":{}, "skipped": skipped,
                                        "forecast_value": latest, "latest_price": latest})
            context["explainability"]["full_trace"].append("[Forecast] All models failed/skipped.")
            return context

        # ✅ ENHANCED MODEL SELECTION using Composite Score
        model_scores = {}
        for model_name, model_metrics in metrics.items():
            score = composite_score(model_metrics, latest)
            model_scores[model_name] = score

        best = max(model_scores.items(), key=lambda x: x[1])[0]
        fval = results[best]
        best_metrics = metrics[best]

        # Enhanced confidence calculation using directional accuracy
        directional_conf = best_metrics["Directional_Accuracy"]
        rmse_conf = max(0.0, min(1.0, 1 - (best_metrics["RMSE"] / latest)))
        conf = float(0.6 * directional_conf + 0.4 * rmse_conf)

        move = "up" if fval>latest else "down" if fval<latest else "stable"

        context["forecast"].update({
            "model_used": best, "predicted_movement": move, "confidence": round(conf,3),
            "metrics_per_model": metrics, "skipped": skipped,
            "forecast_value": fval, "latest_price": latest
        })

        context["explainability"]["full_trace"].append(
            f"[Forecast] {best} → {move.upper()} (conf={conf:.3f}) | "
            f"Dir_Acc={best_metrics['Directional_Accuracy']:.1%}, "
            f"RMSE=${best_metrics['RMSE']:,.2f}, Composite={model_scores[best]:.3f}"
        )
        return context

class SentimentAgent(AgentBase):
    def __init__(self):
        self.api_key = os.environ.get("NEWS_API_KEY","")

    def calculate_sentiment_directional_accuracy(self, context):
        """Calculate historical sentiment directional accuracy"""
        sentiment_history = context.get("sentiment_history", [])

        if len(sentiment_history) < 2:
            return 0.65  # Default reasonable accuracy for bootstrap

        correct_predictions = 0
        total_predictions = 0

        for entry in sentiment_history[-10:]:  # Last 10 predictions
            if all(k in entry for k in ["sentiment_direction", "actual_direction"]):
                total_predictions += 1
                if entry["sentiment_direction"] == entry["actual_direction"]:
                    correct_predictions += 1

        if total_predictions == 0:
            return 0.65

        return round(correct_predictions / total_predictions, 3)

    def categorize_sentiment_strength(self, polarity_score):
        """Categorize sentiment strength based on absolute polarity"""
        abs_score = abs(polarity_score)
        if abs_score >= 0.6:
            return "Strong"
        elif abs_score >= 0.2:
            return "Moderate"
        else:
            return "Weak"

    def calculate_recency_score(self, articles):
        """Calculate how fresh the news is (0-1 scale)"""
        if not articles:
            return 0.0

        now = datetime.now()
        recency_scores = []

        for article in articles:
            published_at = article.get('publishedAt')
            if not published_at:
                continue

            try:
                pub_time = datetime.fromisoformat(published_at.replace('Z', '+00:00'))
                pub_time = pub_time.replace(tzinfo=None)

                hours_ago = (now - pub_time).total_seconds() / 3600

                if hours_ago <= 6:
                    score = 1.0
                elif hours_ago <= 24:
                    score = 0.8
                elif hours_ago <= 48:
                    score = 0.5
                else:
                    score = 0.1

                recency_scores.append(score)
            except:
                recency_scores.append(0.3)

        return round(sum(recency_scores) / len(recency_scores) if recency_scores else 0.3, 3)

    def run(self, context: Dict[str, Any]) -> Dict[str, Any]:
        if not context["sentiment"].get("enabled", False): return context
        symbol = context["data"]["symbol"]

        if not self.api_key:
            # Enhanced neutral fallback with all metrics
            context["sentiment"].update({
                "enabled": True, "source": "NewsAPI", "score": 0.0, "verdict": "neutral",
                "confidence": 0.0, "used_in_decision": False, "alignment": None, "sample_headlines": [],
                "strength_category": "Weak", "news_volume": 0, "recency_score": 0.0,
                "directional_accuracy": 0.65
            })
            context["explainability"]["full_trace"].append("[Sentiment] No NEWS_API_KEY → NEUTRAL.")
            return context

        try:
            from newsapi import NewsApiClient
            from textblob import TextBlob
            cli = NewsApiClient(api_key=self.api_key)

            # Enhanced date range for news search
            if context["data"].get("timestamp"):
                asof_ts = pd.to_datetime(context["data"]["timestamp"]).tz_localize(None).normalize()
            else:
                asof_ts = pd.Timestamp.today().tz_localize(None).normalize()
            frm = (asof_ts - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
            to  = (asof_ts + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

            q = _news_query(symbol)
            arts = cli.get_everything(q=q, language="en", from_param=frm, to=to,
                                    sort_by="publishedAt", page_size=10)

            articles_list = arts.get("articles", [])
            titles = [{"title": a.get("title",""), "source": (a.get("source") or {}).get("name",""), "url": a.get("url", "")}
                      for a in articles_list if a.get("title")]

            if not titles:
                context["sentiment"].update({
                    "enabled": True, "source": "NewsAPI", "score": 0.0, "verdict": "neutral",
                    "confidence": 0.0, "used_in_decision": False, "alignment": None, "sample_headlines": [],
                    "strength_category": "Weak", "news_volume": 0, "recency_score": 0.0,
                    "directional_accuracy": 0.65
                })
                context["explainability"]["full_trace"].append("[Sentiment] No headlines → NEUTRAL.")
                return context

            # Enhanced sentiment analysis
            pols = [TextBlob(t["title"]).sentiment.polarity for t in titles]
            avg = float(sum(pols)/len(pols))
            std = float(np.std(pols))

            # Advanced metrics calculation
            strength_category = self.categorize_sentiment_strength(avg)
            news_volume = len(titles)
            recency_score = self.calculate_recency_score(articles_list)
            directional_accuracy = self.calculate_sentiment_directional_accuracy(context)

            # Enhanced confidence calculation
            base_confidence = 1.0 - std
            volume_boost = min(0.2, news_volume / 50)
            recency_boost = recency_score * 0.15

            enhanced_confidence = round(
                0.6 * base_confidence + 0.3 * volume_boost + 0.1 * recency_boost, 3
            )
            enhanced_confidence = max(0.0, min(1.0, enhanced_confidence))

            verdict = "bullish" if avg > 0.2 else "bearish" if avg < -0.2 else "neutral"
            sample_headlines = titles[:3]

            # Update context with all enhanced metrics
            context["sentiment"].update({
                "enabled": True, "source": "NewsAPI", "score": round(avg,3), "verdict": verdict,
                "confidence": enhanced_confidence, "used_in_decision": False, "alignment": None,
                "sample_headlines": sample_headlines,
                "strength_category": strength_category, "news_volume": news_volume,
                "recency_score": recency_score, "directional_accuracy": directional_accuracy
            })

            context["explainability"]["full_trace"].append(
                f"[Sentiment] Analyzed {news_volume} '{q}' headlines: "
                f"Avg polarity = {avg:.3f} ({strength_category} strength), "
                f"Recency = {recency_score:.1%}, Dir_Acc = {directional_accuracy:.1%} "
                f"→ {verdict.upper()} (Confidence={enhanced_confidence:.1%})"
            )
            return context

        except Exception as e:
            context["sentiment"].update({
                "enabled": True, "source": "NewsAPI", "score": 0.0, "verdict": "neutral",
                "confidence": 0.0, "used_in_decision": False, "alignment": None, "sample_headlines": [],
                "strength_category": "Weak", "news_volume": 0, "recency_score": 0.0,
                "directional_accuracy": 0.65
            })
            context["explainability"]["full_trace"].append(f"[Sentiment] Error → NEUTRAL. {str(e).splitlines()[0][:120]}")
            return context

class DecisionAgent(AgentBase):
    def run(self, context: Dict[str, Any], base_threshold: float = 0.70) -> Dict[str, Any]:
        fc, st = context["forecast"], context["sentiment"]
        move = fc.get("predicted_movement","stable"); conf = float(fc.get("confidence",0.0))
        sv = st.get("verdict","neutral") if st.get("enabled") else "neutral"

        # FIXED: Enhanced threshold adjustment logic (neutral bug fixed)
        if move == "up":
            aligned = (sv == "bullish")
            thr = base_threshold - 0.10 if aligned else (base_threshold + 0.10 if sv == "bearish" else base_threshold)
        elif move == "down":
            aligned = (sv == "bearish")
            thr = base_threshold - 0.10 if aligned else (base_threshold + 0.10 if sv == "bullish" else base_threshold)
        else:
            aligned = None; thr = max(0.75, base_threshold)

        # FIXED: Neutral sentiment alignment logic
        if sv == "neutral":
            aligned = None  # No alignment with neutral sentiment

        if move == "up" and conf >= thr:
            action = "buy"
            rationale = f"Forecast {move.upper()} with confidence {conf:.3f} (threshold {thr:.2f}); sentiment={sv}."
        elif move == "down" and conf >= thr:
            action = "sell"
            rationale = f"Forecast {move.upper()} with confidence {conf:.3f} (threshold {thr:.2f}); sentiment={sv}."
        else:
            reason = "confidence too low" if conf < thr else "no directional edge"
            action = "hold"
            rationale = f"Holding: {reason} (conf {conf:.3f} vs thresh {thr:.2f}); sentiment={sv}."

        context["decision"].update({"action": action, "rationale": rationale, "score": round(conf,3)})
        context["sentiment"]["used_in_decision"] = bool(st.get("enabled"))
        context["sentiment"]["alignment"] = aligned

        # Enhanced history tracking with all new sentiment metrics
        context["history"].append({
            "timestamp": context["data"].get("timestamp"),
            "forecast_movement": move, "forecast_confidence": conf,
            "sentiment_verdict": sv, "sentiment_score": st.get("score"),
            "sentiment_confidence": st.get("confidence"),
            "sentiment_strength": st.get("strength_category"),
            "news_volume": st.get("news_volume"),
            "recency_score": st.get("recency_score"),
            "alignment": aligned, "used_sentiment": st.get("enabled", False)
        })

        context["explainability"]["full_trace"].append(f"[Decision] {action.upper()} → {rationale}")
        return context

class OrchestratorChain:
    def __init__(self):
        self.data_fetcher = DataFetcherAgent()
        self.forecaster   = ForecastAgent()
        self.sentimenter  = SentimentAgent()
        self.decider      = DecisionAgent()

    def run(self, context: Dict[str, Any], base_threshold: float = 0.70) -> Dict[str, Any]:
        try: context = self.data_fetcher.run(context);   context["explainability"]["full_trace"].append("[Chain] DataFetcher completed")
        except Exception as e: context["explainability"]["full_trace"].append(f"[Chain] DataFetcher error: {e}")
        try: context = self.forecaster.run(context);     context["explainability"]["full_trace"].append("[Chain] ForecastAgent completed")
        except Exception as e: context["explainability"]["full_trace"].append(f"[Chain] ForecastAgent error: {e}")
        try: context = self.sentimenter.run(context);    context["explainability"]["full_trace"].append("[Chain] SentimentAgent completed")
        except Exception as e: context["explainability"]["full_trace"].append(f"[Chain] SentimentAgent error: {e}")
        try: context = self.decider.run(context, base_threshold=base_threshold); context["explainability"]["full_trace"].append("[Chain] DecisionAgent completed")
        except Exception as e: context["explainability"]["full_trace"].append(f"[Chain] DecisionAgent error: {e}")
        return context

def MCP_tradi_win_runner(context: Dict[str, Any], use_sentiment: bool = True, base_threshold: float = 0.70) -> Dict[str, Any]:
    context["sentiment"]["enabled"] = use_sentiment
    return OrchestratorChain().run(context, base_threshold=base_threshold)

Overwriting tradiwin_core/core.py


In [18]:
#Cell 11
%%writefile streamlit_app.py
# --- PROFESSIONAL TRADI-WINNING • Complete Enhanced Crypto Trading Assistant ---
# Fixed Version: ALL CRITICAL ISSUES RESOLVED including button visibility and directional accuracy

import os, json, numpy as np, pandas as pd, streamlit as st, altair as alt
from datetime import datetime, timedelta
import importlib, sys

# =========================
# Enhanced Core Import
# =========================
try:
    if "tradiwin_core" in sys.modules:
        import tradiwin_core
        importlib.reload(tradiwin_core)
    else:
        import tradiwin_core
    from tradiwin_core import (
        OrchestratorChain, MCP_tradi_win_runner,
        build_default_context, load_prices, mini_backtest
    )
    core_ok = True
except Exception as e:
    st.error(f"Enhanced core import failed: {e}")
    st.stop()

# =========================
# COMPLETELY FIXED Professional Black Theme with ULTRA VISIBLE Buttons
# =========================
st.set_page_config(
    page_title="TRADI-WINNING",
    layout="wide",
    page_icon="📈",
    initial_sidebar_state="expanded"
)
ART_DIR = "artifacts"; os.makedirs(ART_DIR, exist_ok=True)

# COMPLETELY FIXED CSS with ULTRA VISIBLE buttons and readable everything
st.markdown("""
<style>
/* Main App Background */
.stApp {
    background-color: #0a0a0a !important;
    color: #ffffff !important;
}

/* FIXED Sidebar - Completely Readable */
.css-1d391kg, .css-1cypcdb, section[data-testid="stSidebar"] {
    background-color: #1a1a1a !important;
    border-right: 2px solid #444444 !important;
}

.css-1d391kg .stSelectbox label,
.css-1d391kg .stDateInput label,
.css-1d391kg .stSlider label,
.css-1d391kg .stCheckbox label,
.css-1d391kg h1, .css-1d391kg h2, .css-1d391kg h3,
section[data-testid="stSidebar"] label,
section[data-testid="stSidebar"] h1,
section[data-testid="stSidebar"] h2,
section[data-testid="stSidebar"] h3,
section[data-testid="stSidebar"] p,
section[data-testid="stSidebar"] span,
section[data-testid="stSidebar"] div {
    color: #ffffff !important;
}

/* COMPLETELY FIXED Tooltips - Now Visible and Readable */
div[role="tooltip"],
div[data-baseweb="tooltip"],
.stTooltip,
[data-testid="stTooltipHoverTarget"] + div,
[data-testid="stTooltipContent"] {
    background-color: #000000 !important;
    color: #ffffff !important;
    border: 2px solid #3B82F6 !important;
    border-radius: 8px !important;
    padding: 12px 16px !important;
    font-size: 14px !important;
    font-weight: 600 !important;
    box-shadow: 0 8px 32px rgba(59, 130, 246, 0.5) !important;
    z-index: 999999 !important;
    max-width: 350px !important;
    position: relative !important;
    visibility: visible !important;
    opacity: 1 !important;
}

div[role="tooltip"] *,
div[data-baseweb="tooltip"] *,
.stTooltip *,
[data-testid="stTooltipContent"] * {
    color: #ffffff !important;
    font-weight: 600 !important;
}

/* Force tooltip visibility */
.stTooltip {
    display: block !important;
    visibility: visible !important;
}

/* Main Content */
.main .block-container {
    background-color: #0a0a0a !important;
    padding-top: 2rem !important;
}

/* All Text Elements */
h1, h2, h3, h4, h5, h6, p, span, div, label, .stMarkdown {
    color: #ffffff !important;
}

/* FIXED Select Boxes and Dropdowns */
.stSelectbox > div > div,
.stSelectbox > div > div > div,
.stSelectbox [role="listbox"],
.stSelectbox [role="option"] {
    background-color: #1a1a1a !important;
    color: #ffffff !important;
    border: 1px solid #444444 !important;
}

.stSelectbox [role="option"]:hover {
    background-color: #3B82F6 !important;
    color: #ffffff !important;
}

/* Input Fields */
.stTextInput input, .stNumberInput input, .stDateInput input {
    background-color: #1a1a1a !important;
    color: #ffffff !important;
    border: 1px solid #444444 !important;
    border-radius: 8px !important;
}

/* ULTRA FIXED Main Buttons - Now SUPER VISIBLE */
.stButton > button {
    background: linear-gradient(135deg, #FF6B35 0%, #F7931E 50%, #FFD700 100%) !important;
    color: #000000 !important;
    border: 3px solid #FFD700 !important;
    border-radius: 15px !important;
    padding: 20px 30px !important;
    font-weight: 900 !important;
    font-size: 18px !important;
    text-transform: uppercase !important;
    letter-spacing: 1px !important;
    text-shadow: 1px 1px 2px rgba(0,0,0,0.8) !important;
    transition: all 0.3s ease !important;
    width: 100% !important;
    height: auto !important;
    min-height: 60px !important;
    text-align: center !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    box-shadow: 0 6px 20px rgba(255, 215, 0, 0.4), inset 0 1px 0 rgba(255,255,255,0.2) !important;
    position: relative !important;
    overflow: visible !important;
}

.stButton > button::before {
    content: "🚀 " !important;
    font-size: 20px !important;
    margin-right: 8px !important;
}

.stButton > button:hover {
    background: linear-gradient(135deg, #FFD700 0%, #FFA500 50%, #FF6B35 100%) !important;
    transform: translateY(-3px) scale(1.02) !important;
    box-shadow: 0 12px 30px rgba(255, 215, 0, 0.6), inset 0 1px 0 rgba(255,255,255,0.4) !important;
    border: 3px solid #FFA500 !important;
}

.stButton > button:active {
    transform: translateY(-1px) scale(1.01) !important;
    box-shadow: 0 6px 15px rgba(255, 215, 0, 0.8) !important;
}

/* ULTRA FIXED Download Buttons - Now SUPER VISIBLE */
.stDownloadButton > button {
    background: linear-gradient(135deg, #10B981 0%, #059669 50%, #047857 100%) !important;
    color: #ffffff !important;
    border: 3px solid #10B981 !important;
    border-radius: 15px !important;
    padding: 16px 24px !important;
    font-weight: 800 !important;
    font-size: 16px !important;
    text-transform: uppercase !important;
    letter-spacing: 0.5px !important;
    transition: all 0.3s ease !important;
    width: 100% !important;
    height: auto !important;
    min-height: 50px !important;
    text-align: center !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    box-shadow: 0 4px 15px rgba(16, 185, 129, 0.4), inset 0 1px 0 rgba(255,255,255,0.2) !important;
    position: relative !important;
    overflow: visible !important;
}

.stDownloadButton > button::before {
    content: "📥 " !important;
    font-size: 16px !important;
    margin-right: 6px !important;
}

.stDownloadButton > button:hover {
    background: linear-gradient(135deg, #059669 0%, #047857 50%, #065F46 100%) !important;
    transform: translateY(-2px) scale(1.02) !important;
    box-shadow: 0 8px 25px rgba(16, 185, 129, 0.6), inset 0 1px 0 rgba(255,255,255,0.3) !important;
    border: 3px solid #059669 !important;
}

.stDownloadButton > button:active {
    transform: translateY(0px) scale(1.01) !important;
}

/* FIXED Tables - Centered and Readable */
.dataframe {
    background-color: #1a1a1a !important;
    color: #ffffff !important;
    border: 1px solid #444444 !important;
    border-radius: 8px !important;
    width: 100% !important;
}

.dataframe th {
    background-color: #2d2d2d !important;
    color: #ffffff !important;
    text-align: center !important;
    font-weight: bold !important;
    padding: 12px 8px !important;
    border-bottom: 2px solid #3B82F6 !important;
}

.dataframe td {
    background-color: #1a1a1a !important;
    color: #ffffff !important;
    text-align: center !important;
    padding: 10px 8px !important;
    border-bottom: 1px solid #333333 !important;
}

/* Force center alignment for all table content */
table.dataframe th, table.dataframe td {
    text-align: center !important;
}

.dataframe tr:hover {
    background-color: #2d2d2d !important;
}

/* Tabs */
.stTabs [data-baseweb="tab-list"] {
    background-color: #1a1a1a !important;
    border-bottom: 2px solid #333333 !important;
}

.stTabs [data-baseweb="tab"] {
    background-color: #1a1a1a !important;
    color: #ffffff !important;
    border-radius: 8px 8px 0 0 !important;
    padding: 12px 20px !important;
    font-weight: 600 !important;
}

.stTabs [aria-selected="true"] {
    background-color: #3B82F6 !important;
    color: #ffffff !important;
}

/* Expanders */
.streamlit-expanderHeader {
    background-color: #1a1a1a !important;
    color: #ffffff !important;
    border: 1px solid #444444 !important;
    border-radius: 8px !important;
}

.streamlit-expanderContent {
    background-color: #0f0f0f !important;
    border: 1px solid #333333 !important;
    border-radius: 0 0 8px 8px !important;
}

/* Metrics Container */
.metric-container {
    background: linear-gradient(135deg, #1a1a1a 0%, #2d2d2d 100%) !important;
    border: 1px solid #444444 !important;
    border-radius: 12px !important;
    padding: 16px !important;
    margin: 8px 0 !important;
    box-shadow: 0 4px 12px rgba(0,0,0,0.3) !important;
}

.metric-value {
    font-size: 2rem !important;
    font-weight: bold !important;
    color: #3B82F6 !important;
}

.metric-label {
    font-size: 0.9rem !important;
    color: #cccccc !important;
    margin-bottom: 4px !important;
}

/* Alerts */
.stAlert {
    background-color: #1a1a1a !important;
    border: 1px solid #444444 !important;
    border-radius: 8px !important;
    color: #ffffff !important;
}

/* Status Colors */
.status-success { color: #10B981 !important; font-weight: bold !important; }
.status-warning { color: #F59E0B !important; font-weight: bold !important; }
.status-error { color: #EF4444 !important; font-weight: bold !important; }
.status-info { color: #3B82F6 !important; font-weight: bold !important; }

/* Slider */
.stSlider > div > div > div > div {
    color: #ffffff !important;
}

/* Progress and Spinner */
.stProgress > div > div > div {
    background-color: #3B82F6 !important;
}

.stSpinner > div {
    border-top-color: #3B82F6 !important;
}
</style>
""", unsafe_allow_html=True)

# =========================
# FIXED Enhanced Directional Accuracy Calculation
# =========================
def enhanced_directional_accuracy(y_true, y_pred, min_change_pct=0.1):
    """
    IMPROVED directional accuracy calculation with much better noise filtering
    """
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)

    if len(y_true) < 2 or len(y_pred) < 2:
        return 0.0

    min_len = min(len(y_true), len(y_pred))
    y_true = y_true[:min_len]
    y_pred = y_pred[:min_len]

    if min_len < 2:
        return 0.0

    # IMPROVED: Use percentage changes with much better noise filtering
    actual_changes = np.diff(y_true) / y_true[:-1] * 100
    predicted_changes = np.diff(y_pred) / y_pred[:-1] * 100

    # IMPROVED: Much lower noise threshold for better signal detection
    significant_moves = np.abs(actual_changes) >= min_change_pct

    if not significant_moves.any():
        return 0.5  # Neutral when no significant movements

    filtered_actual = actual_changes[significant_moves]
    filtered_predicted = predicted_changes[significant_moves]

    actual_directions = np.sign(filtered_actual)
    predicted_directions = np.sign(filtered_predicted)

    correct_predictions = (actual_directions == predicted_directions).sum()
    total_predictions = len(filtered_actual)

    if total_predictions == 0:
        return 0.5

    return float(correct_predictions / total_predictions)

# =========================
# FIXED Mini Backtest with Consistent Directional Accuracy
# =========================
def improved_mini_backtest(close_df: pd.DataFrame, asof_ts: pd.Timestamp, lookback_days: int = 30) -> pd.DataFrame:
    """
    FIXED mini backtest using consistent directional accuracy calculation
    """
    df = close_df.copy()
    if df.empty:
        return pd.DataFrame()

    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None).dt.normalize()
    asof_ts = pd.Timestamp(asof_ts).normalize()
    df = df[df["Date"] <= asof_ts].copy()

    if len(df) < 60:
        return pd.DataFrame()

    tail = df.tail(lookback_days + 1).reset_index(drop=True)
    rows = []
    all_predictions = []
    all_actuals = []

    # Build all predictions first
    for i in range(len(tail) - 1):
        cut_date = pd.Timestamp(tail["Date"].iloc[i]).normalize()
        hist = df[df["Date"] <= cut_date]["Close"].to_numpy()

        try:
            from statsmodels.tsa.arima.model import ARIMA
            f = float(ARIMA(hist, order=(3,1,0)).fit().forecast()[0])
        except Exception:
            f = float(hist[-1])

        actual_next = float(tail.loc[i+1, "Close"])
        all_predictions.append(f)
        all_actuals.append(actual_next)

    # FIXED: Use enhanced directional accuracy for the entire series
    overall_dir_acc = enhanced_directional_accuracy(all_actuals, all_predictions)

    # Build individual rows
    for i in range(len(tail) - 1):
        cut_date = pd.Timestamp(tail["Date"].iloc[i]).normalize()
        hist = df[df["Date"] <= cut_date]["Close"].to_numpy()

        try:
            f = float(ARIMA(hist, order=(3,1,0)).fit().forecast()[0])
        except Exception:
            f = float(hist[-1])

        next_day = pd.Timestamp(tail["Date"].iloc[i+1]).normalize()
        actual_next = float(tail.loc[i+1, "Close"])
        latest = float(hist[-1])

        # FIXED: Use forecast-style confidence (consistent with other metrics)
        price_error_pct = abs(f - actual_next) / actual_next
        rmse_conf = max(0.0, min(1.0, 1 - price_error_pct))
        confidence = 0.6 * overall_dir_acc + 0.4 * rmse_conf  # Same as forecast confidence

        # Enhanced directional accuracy for individual prediction
        pred_dir = "UP" if f > latest else "DOWN" if f < latest else "STABLE"
        actual_dir = "UP" if actual_next > latest else "DOWN" if actual_next < latest else "STABLE"
        dir_correct = (pred_dir == actual_dir)

        rows.append({
            "date": next_day.date(),
            "predicted_close": f,
            "actual_close": actual_next,
            "pred_direction": pred_dir,
            "actual_direction": actual_dir,
            "directional_correct": dir_correct,
            "conf": confidence  # FIXED: Now consistent with other confidence calculations
        })

    return pd.DataFrame(rows)

# =========================
# Enhanced Controls with READABLE Sidebar
# =========================
with st.sidebar:
    st.markdown('<h2 style="color: #ffffff !important;">🎛️ Trading Controls</h2>', unsafe_allow_html=True)

    # Cryptocurrency mapping for clear display
    CRYPTO_MAP = {
        "Bitcoin (BTC)": "BTC",
        "Ethereum (ETH)": "ETH",
        "Solana (SOL)": "SOL",
        "Ripple (XRP)": "XRP",
        "Cardano (ADA)": "ADA",
        "BNB Chain (BNB)": "BNB",
        "Polygon (MATIC)": "MATIC",
        "Polkadot (DOT)": "DOT",
        "Chainlink (LINK)": "LINK",
        "Avalanche (AVAX)": "AVAX",
        "Uniswap (UNI)": "UNI",
        "Litecoin (LTC)": "LTC",
        "Cosmos (ATOM)": "ATOM",
        "Fantom (FTM)": "FTM",
        "Algorand (ALGO)": "ALGO"
    }

    selected_crypto = st.selectbox(
        "Select Cryptocurrency",
        list(CRYPTO_MAP.keys()),
        help="Choose the cryptocurrency for analysis. All prices are quoted in USD. Analysis includes quantitative forecasting and qualitative sentiment analysis."
    )
    symbol = CRYPTO_MAP[selected_crypto]
    crypto_full_name = selected_crypto.split(" (")[0]

    df_full = load_prices(symbol)
    if df_full.empty:
        st.error("❌ No price data found. Try a different cryptocurrency or date.")
        st.stop()

    min_d, max_d = df_full["Date"].min(), df_full["Date"].max()
    asof = st.date_input(
        "Analysis Date",
        value=max_d.date(),
        min_value=min_d.date(),
        max_value=max_d.date(),
        help="Analysis includes data up to this date. News sentiment will be filtered around this date (±1 day) for relevance."
    )
    asof_ts = pd.to_datetime(asof).tz_localize(None).normalize()

    # Threshold as percentage
    base_thresh_pct = st.slider(
        "Decision Threshold",
        50, 90, 70, 1,
        help="Minimum forecast confidence percentage for trading. When forecast and sentiment ALIGN, threshold decreases by 10%. When they CONFLICT, threshold increases by 10%. This reduces false signals and improves risk management."
    )
    base_thresh = base_thresh_pct / 100.0

    # Advanced options
    with st.expander("⚙️ Advanced Options"):
        show_advanced_metrics = st.checkbox(
            "Show Advanced Metrics",
            value=True,
            help="Display directional accuracy, sentiment strength analysis, news volume metrics, recency scoring, and composite model selection details."
        )
        enable_debug_mode = st.checkbox(
            "Debug Mode",
            value=False,
            help="Show detailed execution traces, directional accuracy calculations, intermediate steps, and system diagnostics for troubleshooting."
        )

    run = st.button(
        "🚀 Run Enhanced Analysis",
        help="Execute the complete MCP + Agentic pipeline: Data Fetch → Multi-Model Forecasting → Advanced Sentiment Analysis → Intelligent Decision Making with full explainability."
    )

# =========================
# BIGGER TRADI-WINNING TITLE with Crypto Info
# =========================
st.markdown(f"""
<div style="background: linear-gradient(135deg, #1a1a1a 0%, #2d2d2d 100%); border: 2px solid #3B82F6; border-radius: 20px; padding: 24px; margin-bottom: 20px; text-align: center;">
    <div style="margin-bottom: 16px;">
        <h1 style="font-size: 4rem; font-weight: 900; margin: 0; background: linear-gradient(135deg, #3B82F6, #10B981); -webkit-background-clip: text; -webkit-text-fill-color: transparent; letter-spacing: 3px;">
            TRADI-WINNING
        </h1>
        <p style="font-size: 1.3rem; margin: 8px 0 0 0; color: #cccccc; font-style: italic;">
            "tradi-winnin' — helping you make smart crypto decisions"
        </p>
    </div>
    <div style="display: flex; justify-content: center; align-items: center; gap: 20px; flex-wrap: wrap;">
        <span style="padding: 8px 16px; background: #3B82F6; color: white; border-radius: 20px; font-weight: 600; font-size: 0.9rem;">🤖 MCP + AGENTIC</span>
        <span style="padding: 8px 16px; background: #10B981; color: white; border-radius: 20px; font-weight: 600; font-size: 0.9rem;">🛡️ EXPLAINABLE AI</span>
        <span style="padding: 8px 16px; background: #F59E0B; color: white; border-radius: 20px; font-weight: 600; font-size: 0.9rem;">🎯 DIRECTIONAL ACCURACY</span>
    </div>
    <div style="margin-top: 16px; font-size: 1.1rem; color: #ffffff; border-top: 1px solid #444444; padding-top: 12px;">
        <strong>Analyzing:</strong> {crypto_full_name} ({symbol}-USD) • <strong>Date:</strong> {asof_ts.date()} • <strong>Threshold:</strong> {base_thresh_pct}%
    </div>
</div>
""", unsafe_allow_html=True)

# =========================
# Enhanced Pipeline Execution
# =========================
# Initialize defaults
forecast = {"model_used":"—","predicted_movement":"—","confidence":0.0,"metrics_per_model":{}, "skipped":[]}
sentiment = {
    "enabled": False, "source":"NewsAPI", "score":0.0, "verdict":"neutral", "confidence":0.0,
    "sample_headlines":[], "strength_category":"Weak", "news_volume":0, "recency_score":0.0,
    "directional_accuracy":0.65
}
decision = {"action":"—","rationale":"Click 'Run Enhanced Analysis' to generate trading recommendation.","score":0.0}
history = []
trace = None

if run:
    with st.spinner("🔄 Executing Enhanced MCP + Agentic Pipeline..."):
        try:
            # Build enhanced context with debug mode
            ctx = build_default_context(symbol=symbol)
            ctx["sentiment"]["enabled"] = True
            ctx["data"]["timestamp"] = str(asof_ts.date())
            ctx["debug_info"]["enabled"] = enable_debug_mode

            start_time = datetime.now()
            try:
                chain = OrchestratorChain()
                ctx = chain.run(ctx, base_threshold=base_thresh)
            except Exception as fallback_e:
                st.warning(f"⚠️ Chain execution failed, using fallback: {fallback_e}")
                ctx = MCP_tradi_win_runner(ctx, use_sentiment=True)

            execution_time = (datetime.now() - start_time).total_seconds()

            # Extract results
            forecast = ctx.get("forecast", forecast)
            sentiment = ctx.get("sentiment", sentiment)
            decision = ctx.get("decision", decision)
            history = ctx.get("history", [])

            # Enhanced trace with debug info
            trace = {
                "enhanced_metrics": True,
                "execution_timestamp": datetime.utcnow().isoformat(),
                "execution_time_seconds": round(execution_time, 2),
                "cryptocurrency": f"{crypto_full_name} ({symbol})",
                "analysis_date": str(asof_ts.date()),
                "threshold_used": f"{base_thresh_pct}%",
                "debug_info": ctx.get("debug_info", {}) if enable_debug_mode else {}
            }

            if "explainability" in ctx:
                trace["execution_trace"] = ctx["explainability"]["full_trace"]

            st.success(f"✅ Enhanced pipeline completed successfully in {execution_time:.2f} seconds!")

        except Exception as e:
            st.error(f"❌ Pipeline execution error: {e}")
            if enable_debug_mode:
                st.exception(e)

    # Save enhanced artifacts
    for artifact_type, data in {
        "decision": {"cryptocurrency": f"{crypto_full_name} ({symbol})", **decision, "threshold_percentage": f"{base_thresh_pct}%"},
        "forecast": {**forecast, "cryptocurrency": f"{crypto_full_name} ({symbol})"},
        "sentiment": {**sentiment, "cryptocurrency": f"{crypto_full_name} ({symbol})"},
        "trace": trace
    }.items():
        with open(os.path.join(ART_DIR, f"{artifact_type}.json"), "w") as f:
            json.dump(data, f, indent=2, default=str)

# =========================
# Enhanced Professional Tabs
# =========================
tab_overview, tab_forecast, tab_sentiment, tab_correlation, tab_trace = st.tabs([
    "🎯 Overview", "📊 Forecasts", "🧠 Sentiment", "📈 Correlation", "🛡️ Explainability"
])

# -------- ENHANCED OVERVIEW TAB
with tab_overview:
    st.subheader(f"📋 {crypto_full_name} ({symbol}) Trading Decision Overview")

    # Key metrics display
    col1, col2, col3, col4 = st.columns(4)

    with col1:
        action = decision.get("action","—").upper()
        action_color = "#10B981" if action == "BUY" else "#EF4444" if action == "SELL" else "#F59E0B"
        st.markdown(f"""
        <div class="metric-container">
            <div class="metric-label">🎯 Trading Action</div>
            <div class="metric-value" style="color: {action_color};">{action}</div>
        </div>
        """, unsafe_allow_html=True)

    with col2:
        confidence_pct = float(decision.get('score',0)) * 100
        conf_color = "#10B981" if confidence_pct >= 70 else "#F59E0B" if confidence_pct >= 50 else "#EF4444"
        st.markdown(f"""
        <div class="metric-container">
            <div class="metric-label">🎪 Decision Confidence</div>
            <div class="metric-value" style="color: {conf_color};">{confidence_pct:.1f}%</div>
        </div>
        """, unsafe_allow_html=True)

    with col3:
        model_used = forecast.get('model_used','—')
        st.markdown(f"""
        <div class="metric-container">
            <div class="metric-label">🤖 Selected Model</div>
            <div class="metric-value">{model_used}</div>
        </div>
        """, unsafe_allow_html=True)

    with col4:
        movement = forecast.get('predicted_movement','—').upper()
        movement_color = "#10B981" if movement == "UP" else "#EF4444" if movement == "DOWN" else "#6B7280"
        st.markdown(f"""
        <div class="metric-container">
            <div class="metric-label">📈 Price Direction</div>
            <div class="metric-value" style="color: {movement_color};">{movement}</div>
        </div>
        """, unsafe_allow_html=True)

    # FIXED: Decision rationale with percentages
    st.markdown("### 💡 Decision Rationale")
    rationale = decision.get('rationale', 'No analysis completed yet.')
    # Convert confidence values to percentages in rationale
    if 'conf ' in rationale:
        import re
        rationale = re.sub(r'conf (\d+\.\d+)', lambda m: f"conf {float(m.group(1)) * 100:.1f}%", rationale)
    if 'thresh ' in rationale:
        rationale = re.sub(r'thresh (\d+\.\d+)', lambda m: f"thresh {float(m.group(1)) * 100:.1f}%", rationale)
    st.info(rationale)

    # FIXED: Forecast-Sentiment Alignment Explanation (neutral bug fixed)
    if sentiment.get("enabled"):
        alignment_status = sentiment.get('alignment')
        st.markdown("### 🤝 Forecast-Sentiment Alignment Analysis")

        if alignment_status is True:
            st.success("""
            ✅ **ALIGNED** - Forecast and sentiment support each other
            - **Impact**: Trading threshold REDUCED by 10%
            - **Why this helps**: When technical analysis and market sentiment agree, it creates stronger conviction
            - **Best practice**: Reduces false signals and improves risk management
            """)
        elif alignment_status is False:
            st.error("""
            ❌ **CONFLICTING** - Forecast and sentiment oppose each other
            - **Impact**: Trading threshold INCREASED by 10%
            - **Why this helps**: Conflicting signals suggest uncertainty, requiring higher confidence
            - **Best practice**: Prevents trading on weak or contradictory signals
            """)
        else:  # FIXED: This handles neutral correctly now
            st.info("""
            ➖ **NEUTRAL** - No strong sentiment signal detected
            - **Impact**: Standard threshold applied (no adjustment)
            - **Why this happens**: Sentiment is neutral, providing no directional bias
            - **Best practice**: Relies purely on technical forecast without sentiment bias
            """)

    # Enhanced metrics summary
    col1, col2 = st.columns(2)
    with col1:
        forecast_conf_pct = forecast.get('confidence',0) * 100
        st.metric("📊 Forecast Confidence", f"{forecast_conf_pct:.1f}%", help="Model's confidence in price prediction based on historical accuracy and directional prediction strength")

        if forecast.get("metrics_per_model") and forecast.get("model_used"):
            best_model = forecast.get("model_used")
            if best_model in forecast["metrics_per_model"]:
                dir_acc = forecast["metrics_per_model"][best_model].get("Directional_Accuracy", 0) * 100
                st.metric("🎯 Directional Accuracy", f"{dir_acc:.1f}%", help="Historical success rate at predicting correct price direction - the most important metric for trading decisions")

    with col2:
        if sentiment.get("enabled"):
            verdict = sentiment.get('verdict','neutral').upper()
            strength = sentiment.get('strength_category','Weak')
            st.metric("🧠 Sentiment Verdict", f"{verdict}", f"{strength} Strength", help="Market sentiment from news analysis combined with strength assessment")

            news_vol = sentiment.get("news_volume", 0)
            st.metric("📰 News Volume", f"{news_vol} articles", help="Number of relevant news articles analyzed - more articles provide higher confidence in sentiment signal")

    # COMPREHENSIVE Metric Explanations
    with st.expander("📚 Complete Metrics Guide - What Everything Means"):
        st.markdown("""
        ## 🎯 **Trading Decision Metrics**

        ### 🚦 **Trading Action**
        - **BUY**: High confidence upward prediction with favorable risk/reward
        - **SELL**: High confidence downward prediction with favorable risk/reward
        - **HOLD**: Insufficient confidence, conflicting signals, or unfavorable conditions

        ### 🎪 **Decision Confidence vs Forecast Confidence**
        - **Decision Confidence**: Overall confidence in the BUY/SELL/HOLD recommendation (combines forecast + sentiment + alignment)
        - **Forecast Confidence**: Technical model's confidence in price prediction only
        - **Key difference**: Decision confidence includes market sentiment and risk management

        ### 🤖 **Model Selection**
        - Selected using **Composite Score**: 40% Directional Accuracy + 30% MAPE + 30% MAE
        - **ARIMA**: Classical time series, good for trends and momentum
        - **Prophet**: Facebook's model, handles seasonality and market patterns
        - **LSTM**: Neural network, captures complex non-linear relationships

        ### 📈 **Price Direction Prediction**
        - **UP**: Expected price increase based on technical analysis
        - **DOWN**: Expected price decrease based on technical analysis
        - **STABLE**: Minimal significant change expected

        ### 🎯 **Directional Accuracy (Most Important)**
        - **What it measures**: Success rate at predicting correct price direction (up/down)
        - **Why it's critical**: More important than exact price for trading profitability
        - **Calculation**: Compares predicted vs actual direction changes >0.1%
        - **Benchmark**: >70% = Excellent, 50-70% = Good, <50% = Poor

        ### 🧠 **Sentiment Analysis**
        - **Sentiment Confidence**: How much news headlines agree (low disagreement = high confidence)
        - **Strength Categories**: Strong (±60%+), Moderate (±20-60%), Weak (<±20%)
        - **Why it matters**: Market sentiment can override technical signals

        ### 🤝 **Forecast-Sentiment Alignment (Best Practice)**
        - **Aligned**: Both point same direction → Lower threshold (easier to trade)
        - **Conflicting**: Opposite directions → Higher threshold (harder to trade)
        - **Neutral**: No sentiment signal → No threshold adjustment
        - **Why it works**: Reduces false signals, improves risk management, increases win rate
        """)

    # Show skipped models
    skipped = forecast.get("skipped", [])
    if skipped:
        with st.expander("⚠️ Models Skipped Due to Safety Guardrails"):
            st.markdown("**Safety guardrails protect against unreliable predictions:**")
            for model, reason in skipped:
                st.write(f"• **{model}**: {reason}")

# -------- ENHANCED FORECASTS TAB
with tab_forecast:
    st.subheader(f"📊 Advanced Forecast Analysis - {crypto_full_name} ({symbol})")

    # Key forecast metrics
    col1, col2, col3 = st.columns(3)
    with col1:
        movement = forecast.get("predicted_movement","—").upper()
        movement_color = "#10B981" if movement == "UP" else "#EF4444" if movement == "DOWN" else "#6B7280"
        st.markdown(f"""
        <div class="metric-container">
            <div class="metric-label">📈 Predicted Direction</div>
            <div class="metric-value" style="color: {movement_color};">{movement}</div>
        </div>
        """, unsafe_allow_html=True)

    with col2:
        forecast_conf_pct = forecast.get('confidence',0) * 100
        conf_color = "#10B981" if forecast_conf_pct >= 70 else "#F59E0B" if forecast_conf_pct >= 50 else "#EF4444"
        st.markdown(f"""
        <div class="metric-container">
            <div class="metric-label">🎪 Forecast Confidence</div>
            <div class="metric-value" style="color: {conf_color};">{forecast_conf_pct:.1f}%</div>
        </div>
        """, unsafe_allow_html=True)

    with col3:
        if show_advanced_metrics and forecast.get("metrics_per_model"):
            best_model = forecast.get("model_used")
            if best_model and best_model in forecast["metrics_per_model"]:
                dir_acc_pct = forecast["metrics_per_model"][best_model].get("Directional_Accuracy", 0) * 100
                dir_color = "#10B981" if dir_acc_pct >= 70 else "#F59E0B" if dir_acc_pct >= 50 else "#EF4444"
                st.markdown(f"""
                <div class="metric-container">
                    <div class="metric-label">🎯 Directional Accuracy</div>
                    <div class="metric-value" style="color: {dir_color};">{dir_acc_pct:.1f}%</div>
                </div>
                """, unsafe_allow_html=True)

    # Enhanced model performance table with CENTERED values
    mpm = forecast.get("metrics_per_model", {})
    if mpm:
        st.markdown("#### 🏆 Model Performance Comparison")
        st.caption("🎯 **Ranked by Composite Score**: 40% Directional Accuracy + 30% MAPE + 30% MAE")

        # Prepare enhanced table data with FIXED centering
        table_data = []
        selected_model = forecast.get("model_used")

        for model_name, metrics in mpm.items():
            # Calculate composite score (convert to percentage)
            latest_price = forecast.get("latest_price", 50000)
            dir_acc_score = metrics.get("Directional_Accuracy", 0)
            mape_score = max(0, 1 - metrics.get("MAPE", 0) / 100)
            normalized_mae = metrics.get("MAE", 0) / latest_price
            mae_score = max(0, 1 - normalized_mae)
            composite = 0.4 * dir_acc_score + 0.3 * mape_score + 0.3 * mae_score
            composite_pct = composite * 100

            selected_indicator = "👑 " if model_name == selected_model else ""

            table_data.append({
                "Model": f"{selected_indicator}{model_name}",
                "Directional Accuracy": f"{metrics.get('Directional_Accuracy', 0):.2%}",
                "MAPE": f"{metrics.get('MAPE', 0):.2f}%",
                "MAE": f"${metrics.get('MAE', 0):.2f}",
                "RMSE": f"${metrics.get('RMSE', 0):.2f}",
                "Composite Score": f"{composite_pct:.1f}%"
            })

        # Sort by composite score
        table_data.sort(key=lambda x: float(x["Composite Score"].rstrip('%')), reverse=True)

        # Display table with CENTERED values
        df_metrics = pd.DataFrame(table_data)

        # Apply custom styling to ensure centering
        styled_df = df_metrics.style.set_properties(**{
            'text-align': 'center',
            'background-color': '#1a1a1a',
            'color': 'white'
        }).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center'), ('background-color', '#2d2d2d')]},
            {'selector': 'td', 'props': [('text-align', 'center')]}
        ])

        st.dataframe(styled_df, use_container_width=True)

        # FIXED: Formula display with full names
        st.caption("👑 = Selected Model | **Formula**: (40% × Directional Accuracy) + (30% × MAPE Score) + (30% × MAE Score)")

    # ENHANCED: Directional Accuracy Explanation with FIXED example calculation
    with st.expander("🎯 Directional Accuracy - How It's Calculated & Why It's Critical"):
        st.markdown("""
        ## 🎯 **Directional Accuracy: The Most Important Metric**

        ### 📊 **How It's Calculated**
        1. **Take historical price data** (last 30-35 data points for validation)
        2. **Calculate percentage changes** between consecutive periods
        3. **Filter significant moves** (only changes >0.1% to capture more signals)
        4. **Compare directions**:
           - Actual direction: UP if price increased, DOWN if decreased
           - Predicted direction: UP if model predicted increase, DOWN if predicted decrease
        5. **Calculate accuracy**: (Correct predictions ÷ Total predictions) × 100%

        ### 🚀 **Why Directional Accuracy is Critical for Crypto Trading**

        #### **More Important Than Price Precision**
        - **Traditional metrics** (RMSE, MAE) focus on exact price prediction
        - **Directional accuracy** focuses on getting the direction right
        - **Reality**: Being right about direction matters more than exact price precision

        #### **Real Trading Impact**
        - **70%+ accuracy**: Strong edge for profitable trading
        - **60-70% accuracy**: Decent edge with proper risk management
        - **50-60% accuracy**: Marginal edge, need tight risk controls
        - **<50% accuracy**: No trading edge, better to wait

        #### **Why We Weight It 40% in Model Selection**
        - **Traditional approach**: Models selected purely on price accuracy (RMSE)
        - **Our innovation**: Prioritize direction prediction for trading success
        - **Result**: Better trading performance even if exact prices are less precise

        ### 🔍 **Example Calculation**

        **Sample Bitcoin Price Data:**
        ```
        Day 1: $58,000 → Day 2: $59,200 (Change: +2.07%)
        Day 2: $59,200 → Day 3: $58,800 (Change: -0.68%)
        Day 3: $58,800 → Day 4: $60,100 (Change: +2.21%)
        Day 4: $60,100 → Day 5: $59,500 (Change: -1.00%)
        Day 5: $59,500 → Day 6: $61,000 (Change: +2.52%)
        ```

        **Model Predictions:**
        ```
        Predicted Day 2: $59,100 (Change: +1.90%)
        Predicted Day 3: $58,900 (Change: -0.34%)
        Predicted Day 4: $59,900 (Change: +1.70%)
        Predicted Day 5: $59,200 (Change: -1.17%)
        Predicted Day 6: $60,800 (Change: +2.69%)
        ```

        **Direction Comparison:**
        ```
        Actual Directions:    [UP,   DOWN, UP,   DOWN, UP  ]
        Predicted Directions: [UP,   DOWN, UP,   DOWN, UP  ]
        Results:              [✓,    ✓,    ✓,    ✓,    ✓   ]

        Directional Accuracy: 5/5 = 100%
        ```

        **Key Insight:** Even though the exact prices weren't perfect, the model correctly predicted the direction of every move, which is what matters for trading profitability!

        ### 🎪 **Impact on Final Decision**
        - **High directional accuracy** → Higher forecast confidence → More likely to trade
        - **Low directional accuracy** → Lower forecast confidence → More likely to hold
        - **Combined with sentiment** → Even better decision quality
        """)

# -------- ENHANCED SENTIMENT TAB
with tab_sentiment:
    st.subheader(f"🧠 Advanced Sentiment Analysis - {crypto_full_name} ({symbol})")

    if sentiment.get("enabled"):
        # Primary sentiment display
        col1, col2, col3, col4 = st.columns(4)

        with col1:
            verdict = sentiment.get("verdict","neutral").upper()
            strength = sentiment.get("strength_category","Weak")
            verdict_color = "#10B981" if verdict == "BULLISH" else "#EF4444" if verdict == "BEARISH" else "#6B7280"
            st.markdown(f"""
            <div class="metric-container">
                <div class="metric-label">🎯 Market Sentiment</div>
                <div class="metric-value" style="color: {verdict_color};">{verdict}</div>
                <div style="color: #cccccc; font-size: 0.9rem;">({strength} Strength)</div>
            </div>
            """, unsafe_allow_html=True)

        with col2:
            polarity_pct = sentiment.get('score',0) * 100
            polarity_color = "#10B981" if polarity_pct > 20 else "#EF4444" if polarity_pct < -20 else "#6B7280"
            st.markdown(f"""
            <div class="metric-container">
                <div class="metric-label">📊 Polarity Score</div>
                <div class="metric-value" style="color: {polarity_color};">{polarity_pct:+.1f}%</div>
            </div>
            """, unsafe_allow_html=True)

        with col3:
            # FIXED: Confidence naming consistency
            sent_conf_pct = sentiment.get('confidence',0) * 100
            conf_color = "#10B981" if sent_conf_pct >= 70 else "#F59E0B" if sent_conf_pct >= 50 else "#EF4444"
            st.markdown(f"""
            <div class="metric-container">
                <div class="metric-label">🎪 Sentiment Confidence</div>
                <div class="metric-value" style="color: {conf_color};">{sent_conf_pct:.1f}%</div>
            </div>
            """, unsafe_allow_html=True)

        with col4:
            used_status = "✅ YES" if sentiment.get("used_in_decision") else "❌ NO"
            status_color = "#10B981" if sentiment.get("used_in_decision") else "#6B7280"
            st.markdown(f"""
            <div class="metric-container">
                <div class="metric-label">🎯 Used in Decision</div>
                <div class="metric-value" style="color: {status_color};">{used_status}</div>
            </div>
            """, unsafe_allow_html=True)

        # FIXED: Advanced sentiment metrics as separate section
        if show_advanced_metrics:
            st.markdown("---")  # Visual separator
            st.markdown("#### 📊 Advanced Sentiment Performance Metrics")

            col1, col2, col3 = st.columns(3)
            with col1:
                news_vol = sentiment.get("news_volume", 0)
                vol_color = "#10B981" if news_vol >= 5 else "#F59E0B" if news_vol >= 2 else "#EF4444"
                st.markdown(f"""
                <div class="metric-container">
                    <div class="metric-label">📰 News Volume</div>
                    <div class="metric-value" style="color: {vol_color};">{news_vol}</div>
                    <div style="color: #cccccc; font-size: 0.8rem;">articles analyzed</div>
                </div>
                """, unsafe_allow_html=True)

            with col2:
                recency_pct = sentiment.get("recency_score", 0) * 100
                recency_color = "#10B981" if recency_pct >= 70 else "#F59E0B" if recency_pct >= 40 else "#EF4444"
                st.markdown(f"""
                <div class="metric-container">
                    <div class="metric-label">⏰ News Freshness</div>
                    <div class="metric-value" style="color: {recency_color};">{recency_pct:.1f}%</div>
                </div>
                """, unsafe_allow_html=True)

            with col3:
                hist_acc_pct = sentiment.get("directional_accuracy", 0) * 100
                hist_color = "#10B981" if hist_acc_pct >= 70 else "#F59E0B" if hist_acc_pct >= 50 else "#EF4444"
                st.markdown(f"""
                <div class="metric-container">
                    <div class="metric-label">📈 Historical Accuracy</div>
                    <div class="metric-value" style="color: {hist_color};">{hist_acc_pct:.1f}%</div>
                </div>
                """, unsafe_allow_html=True)

        st.markdown("---")  # Visual separator before headlines

        # Enhanced news headlines with sources and dates
        headlines = sentiment.get("sample_headlines", [])
        if headlines:
            st.markdown(f"#### 📰 Recent News Headlines for {crypto_full_name}")
            st.caption(f"Filtered for news around {asof_ts.date()} ± 1 day for relevance")

            for i, headline_data in enumerate(headlines, 1):
                title = headline_data.get('title', 'No title available')
                source = headline_data.get('source', 'Unknown source')
                url = headline_data.get('url', '')

                with st.expander(f"📄 {i}. {title[:80]}{'...' if len(title) > 80 else ''}"):
                    st.write(f"**Full Headline:** {title}")
                    st.write(f"**Source:** {source}")
                    if url:
                        st.write(f"**Link:** [Read full article]({url})")

                    # Individual sentiment analysis
                    try:
                        from textblob import TextBlob
                        blob = TextBlob(title)
                        individual_polarity = blob.sentiment.polarity * 100
                        individual_sentiment = "Positive" if individual_polarity > 10 else "Negative" if individual_polarity < -10 else "Neutral"
                        sent_color = "#10B981" if individual_polarity > 10 else "#EF4444" if individual_polarity < -10 else "#6B7280"
                        st.markdown(f"**Individual Sentiment:** <span style='color: {sent_color}; font-weight: bold;'>{individual_sentiment} ({individual_polarity:+.1f}%)</span>", unsafe_allow_html=True)
                    except:
                        pass

        # COMPREHENSIVE Sentiment Explanations
        with st.expander("📚 Advanced Sentiment Metrics - Complete Guide"):
            st.markdown("""
            ## 🧠 **Complete Sentiment Analysis Guide**

            ### 🎯 **Market Sentiment Verdict**
            - **BULLISH**: Positive news tone (polarity > +20%), market optimism detected
            - **BEARISH**: Negative news tone (polarity < -20%), market pessimism detected
            - **NEUTRAL**: Balanced news tone (-20% to +20%), no clear bias

            ### 📊 **Polarity Score - What The Numbers Mean**
            - **Range**: -100% (extremely negative) to +100% (extremely positive)
            - **Calculation**: Average sentiment polarity across all analyzed headlines
            - **+80% to +100%**: Extremely bullish (rare, very strong positive sentiment)
            - **+40% to +80%**: Strong bullish sentiment
            - **+20% to +40%**: Moderate bullish sentiment
            - **-20% to +20%**: Neutral (no clear direction)
            - **-40% to -20%**: Moderate bearish sentiment
            - **-80% to -40%**: Strong bearish sentiment
            - **-100% to -80%**: Extremely bearish (rare, very strong negative sentiment)

            ### 🎪 **Sentiment Confidence - Why It Matters**
            - **What it measures**: How reliable the sentiment signal is (agreement + volume + recency)
            - **High confidence (70%+)**: Strong, reliable sentiment signal
            - **Medium confidence (50-70%)**: Moderate reliability
            - **Low confidence (<50%)**: Weak or unreliable sentiment signal
            - **Why important**: Confidence determines how much weight sentiment gets in decisions

            ### 📰 **News Volume - Quality Through Quantity**
            - **Why volume matters**: More articles = broader market sentiment, less noise
            - **5+ articles**: Good sample size for reliable analysis
            - **2-4 articles**: Moderate confidence, sufficient for analysis
            - **1 article**: Low confidence, could be outlier/isolated opinion
            - **0 articles**: No sentiment signal available

            ### ⏰ **News Freshness - Timing Is Everything**
            - **0-6 hours**: 100% weight (breaking news, immediate market impact)
            - **6-24 hours**: 80% weight (same day news, still highly relevant)
            - **24-48 hours**: 50% weight (recent but less immediate impact)
            - **48+ hours**: 10% weight (older news, minimal current relevance)
            - **Why crucial**: Crypto markets move fast, old news has less predictive power

            ### 📈 **Historical Accuracy - Learning System**
            - **What it tracks**: How often past sentiment predictions matched actual price movements
            - **Learning mechanism**: System improves accuracy assessment over time
            - **70%+ accuracy**: Strong sentiment predictive power
            - **50-70% accuracy**: Moderate sentiment value
            - **<50% accuracy**: Weak sentiment signal, rely more on technical analysis
            - **Bootstrap value**: 65% (reasonable baseline for new tracking)

            ### 🎯 **How Sentiment Impacts Trading Decisions**
            - **ALIGNED (Sentiment supports forecast)**:
              - Threshold reduced by 10%
              - Example: 70% threshold becomes 60%
              - Easier to trigger trades when both technical and sentiment agree
            - **CONFLICTING (Sentiment opposes forecast)**:
              - Threshold increased by 10%
              - Example: 70% threshold becomes 80%
              - Harder to trigger trades when signals conflict
            - **NEUTRAL**:
              - No threshold adjustment
              - Pure technical analysis drives decision

            ### 🎨 **Strength Categories - Conviction Levels**
            - **Strong (±60%+)**: High conviction sentiment, major market mood shift
            - **Moderate (±20-60%)**: Standard sentiment levels, normal market psychology
            - **Weak (<±20%)**: Low conviction, minimal sentiment impact expected
            """)
    else:
        st.info("🔒 Sentiment analysis is currently disabled. Enable it to see advanced sentiment metrics and news analysis.")

# -------- ENHANCED CORRELATION TAB
with tab_correlation:
    st.subheader(f"📈 Advanced Correlation Analysis - {crypto_full_name} ({symbol})")

    # FIXED: Enhanced backtest with improved directional accuracy
    bt = improved_mini_backtest(df_full, asof_ts, lookback_days=30)
    if not bt.empty:
        st.markdown("#### 📊 Actual vs Predicted Next-Day Prices (Last 30 Trading Days)")
        st.caption("Shows how well our models predict next-day price movements for backtesting validation")

        # COMPLETELY FIXED: Enhanced chart with ULTRA VISIBLE tooltips
        cf = bt.copy()
        cf_long = pd.melt(cf, id_vars=["date"], value_vars=["actual_close","predicted_close"],
                          var_name="Series", value_name="Price")
        cf_long["Series"] = cf_long["Series"].map({
            "actual_close":"Actual Next-Day Price",
            "predicted_close":"Predicted Next-Day Price"
        })

        # ULTRA FIXED: Enhanced chart with SUPER VISIBLE tooltips
        base_chart = alt.Chart(cf_long).add_selection(
            alt.selection_interval(bind='scales')
        )

        lines = base_chart.mark_line(
            strokeWidth=4,
            point=alt.OverlayMarkDef(size=100, filled=True, strokeWidth=2)
        ).encode(
            x=alt.X("date:T",
                   title="Date",
                   axis=alt.Axis(format="%m/%d", labelColor="#ffffff", titleColor="#ffffff",
                               gridColor="#333333", labelFontSize=12, titleFontSize=14)),
            y=alt.Y("Price:Q",
                   title="Price (USD)",
                   scale=alt.Scale(zero=False),
                   axis=alt.Axis(labelColor="#ffffff", titleColor="#ffffff", format="$.0f",
                               gridColor="#333333", labelFontSize=12, titleFontSize=14)),
            color=alt.Color("Series:N",
                          scale=alt.Scale(domain=["Actual Next-Day Price", "Predicted Next-Day Price"],
                                        range=["#10B981", "#3B82F6"]),
                          legend=alt.Legend(title="Price Series", labelColor="#ffffff", titleColor="#ffffff",
                                          orient="top", titleFontSize=14, labelFontSize=12)),
            tooltip=[
                alt.Tooltip("date:T", title="📅 Date", format="%Y-%m-%d"),
                alt.Tooltip("Price:Q", title="💰 Price", format="$,.0f"),
                alt.Tooltip("Series:N", title="📊 Type")
            ]
        ).properties(
            height=500,
            background="#1a1a1a",
            title=alt.TitleParams(
                text=f"{crypto_full_name} ({symbol}) - Prediction Accuracy Validation",
                color="#ffffff",
                fontSize=18,
                fontWeight="bold"
            )
        ).configure(
            background="#1a1a1a"
        ).configure_axis(
            gridColor="#333333",
            domainColor="#666666"
        ).configure_legend(
            fillColor="#1a1a1a",
            strokeColor="#666666",
            padding=15,
            cornerRadius=8
        ).configure_view(
            fill="#1a1a1a",
            stroke="#444444",
            strokeWidth=2
        ).configure_title(
            color="#ffffff",
            fontSize=18
        )

        st.altair_chart(lines, use_container_width=True)

        # FIXED: Enhanced correlation metrics with consistent calculations
        if show_advanced_metrics:
            st.markdown("#### 📊 Correlation Performance Metrics")

            # FIXED: Calculate metrics consistently
            directional_correct = bt['directional_correct'].sum()
            total_predictions = len(bt)
            backtest_dir_acc = directional_correct / total_predictions if total_predictions > 0 else 0
            avg_conf = bt['conf'].mean()
            price_errors = abs(bt['predicted_close'] - bt['actual_close']) / bt['actual_close'] * 100
            avg_price_error = price_errors.mean()

            col1, col2, col3, col4 = st.columns(4)
            with col1:
                dir_color = "#10B981" if backtest_dir_acc >= 0.7 else "#F59E0B" if backtest_dir_acc >= 0.5 else "#EF4444"
                st.markdown(f"""
                <div class="metric-container">
                    <div class="metric-label">🎯 Directional Accuracy</div>
                    <div class="metric-value" style="color: {dir_color};">{backtest_dir_acc:.1%}</div>
                </div>
                """, unsafe_allow_html=True)

            with col2:
                # FIXED: Correlation confidence explanation
                conf_color = "#10B981" if avg_conf >= 0.7 else "#F59E0B" if avg_conf >= 0.5 else "#EF4444"
                st.markdown(f"""
                <div class="metric-container">
                    <div class="metric-label">🎪 Prediction Confidence</div>
                    <div class="metric-value" style="color: {conf_color};">{avg_conf:.1%}</div>
                </div>
                """, unsafe_allow_html=True)

            with col3:
                error_color = "#10B981" if avg_price_error <= 5 else "#F59E0B" if avg_price_error <= 10 else "#EF4444"
                st.markdown(f"""
                <div class="metric-container">
                    <div class="metric-label">📊 Average Price Error</div>
                    <div class="metric-value" style="color: {error_color};">{avg_price_error:.1f}%</div>
                </div>
                """, unsafe_allow_html=True)

            with col4:
                st.markdown(f"""
                <div class="metric-container">
                    <div class="metric-label">📈 Total Predictions</div>
                    <div class="metric-value">{total_predictions}</div>
                </div>
                """, unsafe_allow_html=True)

        # FIXED: Enhanced table with CENTERED values
        st.markdown("#### 📋 Detailed Prediction Results")

        tv = bt.copy()
        tv["Date"] = pd.to_datetime(tv["date"]).dt.date
        tv["Predicted Price"] = tv["predicted_close"].map(lambda x: f"${x:,.0f}")
        tv["Actual Price"] = tv["actual_close"].map(lambda x: f"${x:,.0f}")
        tv["Prediction Confidence"] = (tv["conf"]*100).map(lambda x: f"{x:.1f}%")
        tv["Direction Correct"] = tv["directional_correct"].map(lambda x: "✅ Correct" if x else "❌ Wrong")
        tv["Predicted Direction"] = tv["pred_direction"].map(lambda x: x.upper())
        tv["Actual Direction"] = tv["actual_direction"].map(lambda x: x.upper())

        display_cols = ["Date", "Predicted Price", "Actual Price", "Predicted Direction", "Actual Direction", "Direction Correct", "Prediction Confidence"]

        # Apply custom styling to ensure centering
        styled_tv = tv[display_cols].style.set_properties(**{
            'text-align': 'center',
            'background-color': '#1a1a1a',
            'color': 'white'
        }).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center'), ('background-color', '#2d2d2d')]},
            {'selector': 'td', 'props': [('text-align', 'center')]}
        ])

        st.dataframe(styled_tv, use_container_width=True)

        # ENHANCED Correlation Explanations with confidence clarification
        with st.expander("📚 Correlation Metrics - Complete Understanding"):
            st.markdown("""
            ## 📈 **Correlation Analysis - What It All Means**

            ### 🎯 **Directional Accuracy (Backtest)**
            - **What it shows**: Historical success rate at predicting next-day price direction using ARIMA model
            - **Calculation**: (Days with correct direction prediction ÷ Total trading days) × 100%
            - **Interpretation**:
              - **70%+**: Excellent predictive power, strong trading edge
              - **60-70%**: Good predictive power, profitable with risk management
              - **50-60%**: Moderate edge, requires careful position sizing
              - **<50%**: No predictive edge, avoid trading

            ### 🎪 **Prediction Confidence (Different from Other Confidences)**
            - **What it measures**: ARIMA model's confidence in individual daily predictions
            - **Calculation**: 60% × Overall Directional Accuracy + 40% × Price Accuracy
            - **Why different**: This measures historical prediction reliability, not current decision confidence
            - **Range**: 0% (no confidence) to 100% (maximum confidence)
            - **Use case**: Shows how confident the model was in its historical predictions

            ### 📊 **Confidence Types Comparison**
            - **Decision Confidence**: Overall confidence in BUY/SELL/HOLD recommendation
            - **Forecast Confidence**: Current model's confidence in today's prediction
            - **Sentiment Confidence**: Reliability of news sentiment signal
            - **Prediction Confidence**: Historical ARIMA model prediction reliability

            ### 📊 **Average Price Error**
            - **What it measures**: How far off historical predictions were in percentage terms
            - **Calculation**: |Predicted Price - Actual Price| ÷ Actual Price × 100%
            - **Benchmark**:
              - **<5%**: Excellent price accuracy
              - **5-10%**: Good price accuracy
              - **10-15%**: Moderate accuracy
              - **>15%**: Poor accuracy, direction more important than precision

            ### 📈 **Chart Interpretation**
            - **Green line**: Actual next-day prices (what really happened)
            - **Blue line**: Predicted next-day prices (ARIMA model forecasts)
            - **Close lines**: Good price prediction accuracy
            - **Similar patterns**: Good direction prediction (more important for trading)
            - **Divergence**: Model struggling with current market conditions

            ### 🎯 **How to Use This Information**

            #### **For Current Decisions**
            - **High recent accuracy**: More confidence in today's recommendation
            - **Low recent accuracy**: Be more cautious, consider waiting
            - **Improving trend**: Model adapting to current market
            - **Declining trend**: Market regime may be changing

            #### **For Risk Management**
            - **Consistent accuracy**: Normal position sizing
            - **Variable accuracy**: Reduce position sizes during uncertain periods
            - **High confidence + high accuracy**: Opportunity for larger positions
            - **Low confidence or accuracy**: Minimal positions or wait

            ### 📅 **30-Day Window Rationale**
            - **Recent performance**: Most relevant for current market conditions
            - **Sufficient data**: Enough trades to be statistically meaningful
            - **Market regime**: Captures current volatility and trend patterns
            - **Adaptive**: Updates daily as new data becomes available
            """)

    else:
        st.info("📊 Insufficient historical data for correlation analysis. Need at least 60 days of price history before the selected analysis date.")

# -------- ENHANCED EXPLAINABILITY TAB
with tab_trace:
    st.subheader("🛡️ Complete System Explainability & Transparency")

    st.markdown(f"""
    **Complete transparency** into every decision made by the TRADI-WINNING system for **{crypto_full_name} ({symbol})**.
    This trace shows exactly how quantitative forecasting, qualitative sentiment analysis, and risk management
    combine through our MCP + Agentic architecture to produce trustworthy, explainable recommendations.
    """)

    if trace:
        # Enhanced execution summary
        st.markdown("#### ⚡ Execution Summary")

        col1, col2, col3, col4 = st.columns(4)
        with col1:
            exec_time = trace.get("execution_time_seconds", 0)
            time_color = "#10B981" if exec_time < 30 else "#F59E0B" if exec_time < 60 else "#EF4444"
            st.markdown(f"""
            <div class="metric-container">
                <div class="metric-label">⏱️ Execution Time</div>
                <div class="metric-value" style="color: {time_color};">{exec_time:.1f}s</div>
            </div>
            """, unsafe_allow_html=True)

        with col2:
            st.markdown(f"""
            <div class="metric-container">
                <div class="metric-label">🪙 Cryptocurrency</div>
                <div class="metric-value">{crypto_full_name}</div>
                <div style="color: #cccccc; font-size: 0.8rem;">({symbol})</div>
            </div>
            """, unsafe_allow_html=True)

        with col3:
            st.markdown(f"""
            <div class="metric-container">
                <div class="metric-label">📅 Analysis Date</div>
                <div class="metric-value">{trace.get('analysis_date', 'N/A')}</div>
            </div>
            """, unsafe_allow_html=True)

        with col4:
            st.markdown(f"""
            <div class="metric-container">
                <div class="metric-label">🎯 Threshold Used</div>
                <div class="metric-value">{trace.get('threshold_used', 'N/A')}</div>
            </div>
            """, unsafe_allow_html=True)

        # Enhanced trace display
        if "execution_trace" in trace:
            st.markdown("#### 🔍 Detailed Agent Execution Trace")

            trace_lines = trace["execution_trace"]
            for i, line in enumerate(trace_lines, 1):
                # Enhanced formatting based on content
                if "✅" in line or "completed" in line.lower():
                    st.success(f"**Step {i:02d}**: {line}")
                elif "❌" in line or "failed" in line.lower() or "error" in line.lower():
                    st.error(f"**Step {i:02d}**: {line}")
                elif "⚠️" in line or "warning" in line.lower():
                    st.warning(f"**Step {i:02d}**: {line}")
                else:
                    st.info(f"**Step {i:02d}**: {line}")

        # Debug information (if enabled)
        if enable_debug_mode and "debug_info" in trace:
            debug_info = trace["debug_info"]
            if debug_info.get("directional_accuracy_details"):
                st.markdown("#### 🔧 Directional Accuracy Debug Information")

                with st.expander("🔍 Detailed Directional Accuracy Calculations"):
                    for i, detail in enumerate(debug_info["directional_accuracy_details"], 1):
                        st.markdown(f"**Model {i} Calculation:**")
                        st.json(detail)

        # ULTRA FIXED: Enhanced download options with SUPER VISIBLE buttons
        st.markdown("#### 💾 Export Complete Analysis")

        col1, col2 = st.columns(2)
        with col1:
            # Comprehensive trace
            comprehensive_trace = {
                **trace,
                "system_metadata": {
                    "generated_by": "TRADI-WINNING Enhanced System v2.0",
                    "architecture": "MCP + Agentic AI",
                    "cryptocurrency_analyzed": f"{crypto_full_name} ({symbol})",
                    "analysis_date": str(asof_ts.date()),
                    "threshold_percentage": f"{base_thresh_pct}%",
                    "advanced_metrics_enabled": show_advanced_metrics,
                    "debug_mode_enabled": enable_debug_mode,
                    "model_selection_method": "Enhanced Composite Score (40% Directional Accuracy + 30% MAPE + 30% MAE)",
                    "sentiment_analysis": "Advanced Multi-Metric with Recency Weighting",
                    "confidence_calculation": "Directional Accuracy Weighted (60% Dir_Acc + 40% RMSE_Conf)",
                    "explainability_level": "Complete System Transparency"
                }
            }

            st.download_button(
                "📥 Download Complete Trace",
                data=json.dumps(comprehensive_trace, indent=2, default=str),
                file_name=f"TRADI_WINNING_complete_trace_{symbol}_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json",
                mime="application/json",
                help="Complete system execution trace with all advanced metrics, debug information, and comprehensive metadata",
                key="download_trace"
            )

        with col2:
            # Executive summary
            executive_summary = {
                "executive_summary": {
                    "cryptocurrency": f"{crypto_full_name} ({symbol})",
                    "analysis_date": str(asof_ts.date()),
                    "final_recommendation": decision.get("action", "N/A").upper(),
                    "decision_confidence": f"{float(decision.get('score', 0)) * 100:.1f}%",
                    "selected_model": forecast.get("model_used", "N/A"),
                    "forecast_direction": forecast.get("predicted_movement", "N/A").upper(),
                    "forecast_confidence": f"{forecast.get('confidence', 0) * 100:.1f}%",
                    "sentiment_verdict": sentiment.get("verdict", "N/A").upper(),
                    "sentiment_strength": sentiment.get("strength_category", "N/A"),
                    "threshold_used": f"{base_thresh_pct}%",
                    "execution_time": f"{trace.get('execution_time_seconds', 0):.1f}s"
                },
                "key_metrics": {
                    "directional_accuracy": f"{forecast.get('metrics_per_model', {}).get(forecast.get('model_used', ''), {}).get('Directional_Accuracy', 0) * 100:.1f}%",
                    "composite_score": f"{(forecast.get('metrics_per_model', {}).get(forecast.get('model_used', ''), {}).get('Directional_Accuracy', 0) * 0.4 + max(0, 1 - forecast.get('metrics_per_model', {}).get(forecast.get('model_used', ''), {}).get('MAPE', 0) / 100) * 0.3 + max(0, 1 - forecast.get('metrics_per_model', {}).get(forecast.get('model_used', ''), {}).get('MAE', 0) / forecast.get('latest_price', 50000)) * 0.3) * 100:.1f}%",
                    "sentiment_confidence": f"{sentiment.get('confidence', 0) * 100:.1f}%",
                    "news_volume": sentiment.get("news_volume", 0),
                    "news_freshness": f"{sentiment.get('recency_score', 0) * 100:.1f}%",
                    "forecast_sentiment_alignment": "Aligned" if sentiment.get('alignment') is True else "Conflicting" if sentiment.get('alignment') is False else "Neutral"
                }
            }

            st.download_button(
                "📊 Download Executive Summary",
                data=json.dumps(executive_summary, indent=2, default=str),
                file_name=f"TRADI_WINNING_executive_summary_{symbol}_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json",
                mime="application/json",
                help="Executive-level summary with key findings, recommendations, and performance metrics",
                key="download_summary"
            )
    else:
        st.info("🔄 Execute an enhanced analysis to generate a complete explainability trace with advanced metrics.")

# =========================
# PROFESSIONAL FOOTER (FIXED - No flags)
# =========================
st.markdown(f"""
<div style="margin-top: 40px; padding: 24px; background: linear-gradient(135deg, #1a1a1a 0%, #2d2d2d 100%); border: 1px solid #333333; border-radius: 16px; text-align: center;">
    <div style="font-size: 1.2rem; font-weight: 700; margin-bottom: 12px; color: #3B82F6;">
        🚀 TRADI-WINNING v2.0 • Enhanced Professional Crypto Trading Assistant
    </div>
    <div style="font-size: 1rem; margin-bottom: 8px; color: #ffffff;">
        <span style="margin-right: 25px;">🤖 <strong>MCP + Agentic Architecture</strong></span>
        <span style="margin-right: 25px;">🎯 <strong>Directional Accuracy Innovation</strong></span>
        <span>🛡️ <strong>Complete Explainability</strong></span>
    </div>
    <div style="font-size: 0.9rem; color: #cccccc; margin-bottom: 12px;">
        Advanced Sentiment Analysis • Multi-Model Forecasting • Composite Scoring • Professional Grade
    </div>
    <div style="font-size: 0.85rem; color: #ffffff; border-top: 1px solid #444444; padding-top: 12px;">
        Designed and Developed by <strong>Daniel Ojeda Rosales</strong>
    </div>
</div>
""", unsafe_allow_html=True)

Overwriting streamlit_app.py


In [19]:

#Cell 12
!pip -q install streamlit pyngrok

import os, getpass, subprocess, time, socket, sys, signal, psutil, textwrap

# 0) Sanity: do we have the app file?
assert os.path.exists("streamlit_app.py"), "❌ streamlit_app.py not found. Re-run the cell that writes it."

# 1) Kill anything on 8501 + old ngrok/streamlit
def kill_like(name):
    for p in psutil.process_iter(attrs=["pid","name","cmdline"]):
        try:
            cmd = " ".join(p.info.get("cmdline") or [])
            if name in cmd.lower():
                p.kill()
        except Exception:
            pass

import psutil
kill_like("streamlit run")
kill_like("ngrok")
# also free the port if something is stuck
try:
    import psutil, socket
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    if s.connect_ex(("127.0.0.1", 8501)) == 0:
        for p in psutil.process_iter(attrs=["pid","connections"]):
            for c in p.info.get("connections") or []:
                if getattr(c, "laddr", None) and c.laddr.port == 8501:
                    p.kill()
    s.close()
except Exception:
    pass

# 2) Start Streamlit in background, capture logs
env = os.environ.copy()
env["STREAMLIT_SERVER_HEADLESS"] = "true"
env["STREAMLIT_SERVER_PORT"] = "8501"
env["STREAMLIT_SERVER_ADDRESS"] = "0.0.0.0"
env["BROWSER_GATHER_USAGE_STATS"] = "false"
# pass your NewsAPI key from earlier cell if set
env["NEWS_API_KEY"] = os.environ.get("NEWS_API_KEY", "")

log_path = "streamlit.log"
log = open(log_path, "w")
proc = subprocess.Popen(["streamlit", "run", "streamlit_app.py"], stdout=log, stderr=log, env=env)

def wait_for_port(host="127.0.0.1", port=8501, timeout=90):
    end = time.time() + timeout
    while time.time() < end:
        try:
            s = socket.create_connection((host, port), timeout=1)
            s.close()
            return True
        except Exception:
            time.sleep(1)
    return False

if not wait_for_port():
    log.close()
    print("❌ Streamlit did not start on :8501.\n---- Last 200 log lines ----")
    try:
        print("".join(open(log_path).readlines()[-200:]))
    except Exception:
        print("(no log output)")
    raise SystemExit

# 3) Start ngrok after the port is confirmed open
from pyngrok import ngrok, conf
token = getpass.getpass("🔑 Enter your ngrok authtoken: ")
try:
    ngrok.set_auth_token(token)
except Exception:
    conf.get_default().auth_token = token

try:
    tunnel = ngrok.connect(addr=8501, proto="http", bind_tls=True)
except TypeError:
    tunnel = ngrok.connect(addr=8501, proto="http")

print("✅ Public URL:", tunnel.public_url)
print("⚙️  Local URL:  http://127.0.0.1:8501")
print(f"🧾 Logs: tail -n 200 {log_path}")


🔑 Enter your ngrok authtoken: ··········
✅ Public URL: https://30c989e7136a.ngrok-free.app
⚙️  Local URL:  http://127.0.0.1:8501
🧾 Logs: tail -n 200 streamlit.log
